# Segmentation and Tumour Detection in Histopathology Images

Unsupervised and supervised segmentation of nuclei in H&E-stained breast-cancer
histopathology, then a blob-feature detection model that counts tumours per slide.

Built as a team project for ECSE 415 (Computer Vision) at McGill University, Winter 2025.


In [ ]:
# Tried to use sklearnex but wasn't being recognized
# from sklearnex import patch_sklearn
# patch_sklearn()

# Used for everything
import matplotlib.pyplot as plt
import matplotlib.image as mpimg # Not super necessary
import pickle
from pickle import dump, load
from joblib import Parallel, delayed
import cv2
import numpy as np, sys
import math as m
import os
import pandas as pd
import ast
from collections import defaultdict

# Yellowbrick
from yellowbrick.cluster import KElbowVisualizer

# Sklearn
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.utils import class_weight
from sklearn.model_selection import GridSearchCV, train_test_split

# Seaborn
import seaborn as sns

# Skimage
from skimage.feature import hog, local_binary_pattern
from skimage.measure import regionprops, label

# Scipy
from scipy.stats import skew

path = './QC/'

## 1. Dataset

### 1.1. Plotting images for a sample subject

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Function to plot input image, mask, and overlay
def plot_images(image, mask, mask_overlay):
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(image)
    ax[0].set_title('Input Image')
    ax[0].axis('off')
    ax[1].imshow(mask, cmap='gray')
    ax[1].set_title('Mask')
    ax[1].axis('off')
    ax[2].imshow(mask_overlay)
    ax[2].set_title('Mask Overlay on Input Image')
    ax[2].axis('off')
    plt.show()

# Load the image and mask
training_image_path = 'TCGA-A1-A0SP-DX1_id-5ea4095addda5f8398977ebc_left-8066_top-55998_bottom-56282_right-8338'
testing_image_path = 'TCGA-S3-AA15-DX1_id-5ea40a6addda5f839898f24a_left-56766_top-30705_bottom-30988_right-57025'
input_image = cv2.imread(path + 'rgb/' + training_image_path + '.png')
mask = cv2.imread(path + 'mask/' + training_image_path + '.png')
input_image = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
mask_color = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)



In [ ]:
# Function to draw contours and associate nuclei with classifications
def draw_contours_from_csv(image, csv_path):
    df = pd.read_csv(csv_path)
    overlay = image.copy()

    for index, row in df.iterrows():
        coords_x = list(map(int, row['coords_x'].split(',')))
        coords_y = list(map(int, row['coords_y'].split(',')))
        points = np.array(list(zip(coords_x, coords_y)), dtype=np.int32)

        # Assign color for visualization
        color_red = (255, 0, 0)
        color_blue = (0, 0, 255)
        if row['raw_classification'] == 'tumor':
            color = color_red
        else:
            color = color_blue
        # Draw contours
        if row['type'] == "polyline":
            cv2.polylines(overlay, [points], isClosed=True, color=color, thickness=2)
        elif row['type'] == "rectangle":
            xmin, ymin, xmax, ymax = row['xmin'], row['ymin'], row['xmax'], row['ymax']
            cv2.rectangle(overlay, (xmin, ymin), (xmax, ymax), color, 2)

        # Add text label
        label = row['raw_classification']
        cv2.putText(overlay, label, (coords_x[0], coords_y[0] - 5), cv2.FONT_HERSHEY_SIMPLEX, 
                    0.5, color, 1, cv2.LINE_AA)

    return overlay

# Load CSV and generate overlay + classification mapping
csv_path = path + 'csv/' + training_image_path + '.csv'
overlay_image = draw_contours_from_csv(input_image, csv_path)

# Plot the images
plot_images(input_image, mask_color, overlay_image)

### 1.2. Obtain unique subjects and create a training (~80%) and validation split (~20%)

In [ ]:
# Empty the splits folder
def empty_splits():
    for filename in os.listdir(path+'splits'):
        os.remove(path+'splits/'+filename)

# Find the subjects encompassed by the file names - ex: "TCGA-A1-A0SP-DX1_id" is the unique identifier for the subject
def find_subjects(path):
    subjects = []
    for filename in os.listdir(path+'rgb'):
        subjectname = filename.split('_')[0] + '_id'
        if subjectname not in subjects:
            subjects.append(subjectname)
    return list(set(subjects))

# Split 80% of the subjects into the training set and 20% into the testing set
# Training set in "splits/traning_subjects.csv" and testing set in "splits/testing_subjects.csv"
def split_subjects(subjects):
    split = int(len(subjects)*0.8)
    training = subjects[:split]
    testing = subjects[split:]
    with open(path+'splits/train_subjects.csv', 'w') as f:
        f.write('subject_id\n')
        for subject in training:
            f.write(subject+'\n')
    with open(path+'splits/validate_subjects.csv', 'w') as f:
        f.write('subject_id\n')
        for subject in testing:
            f.write(subject+'\n')
    return training, testing

empty_splits()
subjects = find_subjects(path)
training, testing = split_subjects(find_subjects(path))

print('Subjects:', subjects, '\nNumber of subjects',len(subjects))
print('Training:', training, '\nNumber of training subjects', len(training))
print('Testing:', testing, '\nNumber of testing subjects', len(testing))

# Folder containing the RGB images
rgb_dir = os.path.join(path, 'rgb')

# Get all filenames, strip .png extension, and filter only .png files
image_paths = [os.path.splitext(fname)[0] for fname in os.listdir(rgb_dir) if fname.endswith('.png')]

input_csv = path + 'csv/' + training_image_path + '.csv'

## 2. Segmentation

### 2.1. Segmentation - Unsupervised

#### 2.1.1. Find an optimized value of K. Show the results for K=3 and K= 5.

In [ ]:
#Source: https://jakevdp.github.io/PythonDataScienceHandbook/05.11-k-means.html

input_image = cv2.imread(path + 'rgb/' + training_image_path + '.png')
img = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
h, w, ch = img.shape
print('Image shape:', img.shape)
img_flatten = img.reshape(h * w, ch)  # Flatten the image to a 2D array of pixels
print('Flattened image shape:', img_flatten.shape)

# Visualize the images as a 2D scatter plot in RGB space
plt.figure(figsize=(10, 8))
plt.scatter(img_flatten[:, 0], img_flatten[:, 1], c=img_flatten / 255.0, s=1)
plt.xlabel('Red Channel')
plt.ylabel('Green Channel')
plt.title('2D Scatter Plot of RGB Pixels')
plt.show()

# Visualize the image as a 3D scatter plot in RGB space
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(img_flatten[:, 0], img_flatten[:, 1], img_flatten[:, 2], c=img_flatten / 255.0, s=1)
ax.set_xlabel('Red Channel')
ax.set_ylabel('Green Channel')
ax.set_zlabel('Blue Channel')
ax.set_title('3D Scatter Plot of RGB Pixels')
plt.show()

In [ ]:
# Source: https://builtin.com/data-science/elbow-method#:~:text=The%20elbow%20method%20is%20a%20graphical%20method%20for%20finding%20the,the%20graph%20forms%20an%20elbow.
# Find the optimal number of clusters using the Elbow method
def find_optimal_k_elbow(img_flatten, max_k):
    distortions = []
    for k in range(2, max_k + 1):
        kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
        kmeans.fit(img_flatten)
        distortions.append(kmeans.inertia_)  # Sum of squared distances to closest centroid
    
    # Plot Elbow Curve
    plt.figure(figsize=(6, 4))
    plt.plot(range(2, max_k + 1), distortions, marker='o')
    plt.xlabel('Number of Clusters (K)')
    plt.ylabel('Distortion')
    plt.title('Elbow Method for Optimal K')
    plt.show()

    # Using Yellowbrick for a more visual approach => This says K=4 is the best choice but 3 seems better
    km = KMeans(random_state=0, n_init=10)
    visualizer = KElbowVisualizer(km, k=(2,max_k+1)) # Create the visualizer with the KMeans model and the range of K values
    visualizer.fit(img_flatten)        # Fit the data to the visualizer
    visualizer.show()        # Finalize and render the figurer
    return visualizer.elbow_value_

optimal_k = find_optimal_k_elbow(img_flatten, max_k=6) 
print(f'Optimal K (Elbow): {optimal_k}') # Based on the elbow method, we can see that K=3 is a good choice

In [ ]:
#https://www.kaggle.com/code/abrahamanderson/image-segmentation-with-k-means-clustering
# Apply K-means segmentation with K = 3 and K = 5
# Function to perform K-means segmentation
def k_means_segmentation(img, k):
    img_flatten = img.reshape((-1, 3))
    kmeans = KMeans(n_clusters=k, random_state=0, n_init=10).fit(img_flatten)#for kmeans segmentation
    labels = kmeans.labels_
    h, w = img.shape[:2]

    label_map = labels.reshape(h, w)  # For mask creation
    color_segmentation = kmeans.cluster_centers_[labels].reshape(h, w, 3).astype(np.uint8)  # For display
    return label_map, color_segmentation, kmeans.cluster_centers_

label_map_k3, color_seg_k3, centers_k3 = k_means_segmentation(input_image, 3)
label_map_k5, color_seg_k5, centers_k5 = k_means_segmentation(input_image, 5)


fig, ax = plt.subplots(1, 3, figsize=(20, 5))
ax[0].imshow(input_image)
ax[0].set_title('Original Image')
ax[0].axis('off')

ax[1].imshow(color_seg_k3)
ax[1].set_title('Segmentation with K=3')
ax[1].axis('off')

ax[2].imshow(color_seg_k5)
ax[2].set_title('Segmentation with K=5')
ax[2].axis('off')
plt.show()

#### 2.1.2. How did you decide the optimized value of K? Explain your answer in the context of computing DICE coefficients.



In [ ]:
#based on the csv
def draw_shapes_from_csv(input_csv, fill = True):
    df = pd.read_csv(input_csv)

    # Define image dimensions
    ground_truth_segmentation = np.zeros((h, w), dtype=np.uint8)

    for _, row in df.iterrows():
        coords_x = np.array(ast.literal_eval(row['coords_x']), dtype=np.int32)
        coords_y = np.array(ast.literal_eval(row['coords_y']), dtype=np.int32)
        points = np.column_stack((coords_x, coords_y))

        #https://docs.opencv.org/4.x/dc/da5/tutorial_py_drawing_functions.html
        if fill:
            if row["type"] == "polyline":
                # Fill the polygon 
                cv2.fillPoly(ground_truth_segmentation, [points], color=255)
            elif row["type"] == "rectangle":
                xmin, ymin, xmax, ymax = int(row['xmin']), int(row['ymin']), int(row['xmax']), int(row['ymax'])
                # Fill the rectangle
                cv2.rectangle(ground_truth_segmentation, (xmin, ymin), (xmax, ymax), color=255, thickness=-1)
                    # Draw contours
        else:
            if row['type'] == "polyline":
                cv2.polylines(ground_truth_segmentation, [points], isClosed=True, color=255, thickness=2)
            elif row['type'] == "rectangle":
                xmin, ymin, xmax, ymax = row['xmin'], row['ymin'], row['xmax'], row['ymax']
                cv2.rectangle(ground_truth_segmentation, (xmin, ymin), (xmax, ymax), color=255, thickness= 2)

    return ground_truth_segmentation


ground_truth_mask = draw_shapes_from_csv(input_csv)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(input_image)
ax[0].set_title('Input Image')
ax[0].axis('off')

ax[1].imshow(ground_truth_mask, cmap='gray')
ax[1].set_title('Ground Truth Mask')
ax[1].axis('off')

masked = np.ma.masked_where(ground_truth_mask == 0, ground_truth_mask)
ax[2].imshow(input_image)
ax[2].imshow(masked, cmap='jet', alpha=0.5)
ax[2].set_title('Mask Overlay')
ax[2].axis('off')
plt.show()

In [ ]:
from joblib import Parallel, delayed
##https://medium.com/@nghihuynh_37300/understanding-evaluation-metrics-in-medical-image-segmentation-d289a373a3f
# Compute DICE coefficient between two binary masks
def dice_coefficient(pred_mask, true_mask):
    
    pred_mask = (pred_mask > 0).astype(np.uint8)
    true_mask = (true_mask > 0).astype(np.uint8)
    
    intersection = np.sum(pred_mask * true_mask)
    total = np.sum(pred_mask) + np.sum(true_mask)
    
    return (2.0 * intersection) / total if total > 0 else 0.0

# Find optimal K using DICE coefficients
def find_optimal_k_dice(img, true_mask, show_plot, max_k=10):
    
    dice_scores = []
    for k in range(3, max_k + 1):
        labels, h , c = k_means_segmentation(img, k)
        best_dice_k = 0
        best_cluster = 0

        for i in range(k):
            temp_mask = (labels == i).astype(np.uint8) * 255
            temp_mask = cv2.resize(temp_mask, (true_mask.shape[1], true_mask.shape[0]))
            dice = dice_coefficient(temp_mask, true_mask)
            if dice > best_dice_k:
                best_dice_k = dice

        dice_scores.append(best_dice_k)
        
        if show_plot:
            print(f"K={k}: Best DICE = {best_dice_k:.4f}")

        if best_dice_k > max(dice_scores[:-1], default=0): #make sure best value
            best_k = k
    if show_plot:
        # Plot DICE vs K
        plt.figure(figsize=(8, 4))
        plt.plot(range(3, max_k +1), dice_scores, marker='o')
        plt.xlabel('Number of clusters (K)')
        plt.ylabel('DICE Coefficient')
        plt.title('DICE Score vs Number of Clusters')
        plt.grid(True)
        plt.show()

        plt.imshow(temp_mask)
        plt.imshow(true_mask, cmap='jet', alpha=0.5)
        plt.axis('off')
        plt.show()
    
    return best_k, max(dice_scores)

In [ ]:
# Find optimal K
optimal_k, best_dice = find_optimal_k_dice(input_image, ground_truth_mask, True,  max_k=10)
print(f"Optimal K: {optimal_k}")

# Visualize optimal segmentation
label_map_opt, color_seg_opt, centers_opt = k_means_segmentation(input_image, optimal_k)

# Create color segmentation for display
h, w = input_image.shape[:2]
color_seg = centers_opt[label_map_opt].reshape(h, w, 3).astype(np.uint8)

In [ ]:
from joblib import Parallel, delayed

def evaluate_kmeans_on_training_set_single(img_path):
    input_image = cv2.imread(path + 'rgb/' + img_path + '.png')
    input_image = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)

    ground_truth_mask = draw_shapes_from_csv(path + 'csv/' + img_path + '.csv')

    optimal_k, best_dice = find_optimal_k_dice(input_image, ground_truth_mask, False)
    return optimal_k, best_dice


def evaluate_kmeans_on_training_set_parallel(training_images, n_jobs=-1):
    results = Parallel(n_jobs=n_jobs)(
        delayed(evaluate_kmeans_on_training_set_single)(img)
        for img in training_images
    )

    # Unpack results into two separate lists
    optimal_k_l, best_dice_l = zip(*results)
    optimal_k_l = list(optimal_k_l)
    best_dice_l = list(best_dice_l)

    avg_k = np.mean(optimal_k_l)
    min_k = np.min(optimal_k_l)
    max_k_val = np.max(optimal_k_l)

    avg_dice = np.mean(best_dice_l)
    min_dice = np.min(best_dice_l)
    max_dice = np.max(best_dice_l)

    print("\n========== DICE Summary (Parallel) ==========")
    print(f"Average K: {avg_k:.4f}")
    print(f"Min K:     {min_k:.4f}")
    print(f"Max K:     {max_k_val:.4f}")

    print(f"Average DICE: {avg_dice:.4f}")
    print(f"Min DICE:     {min_dice:.4f}")
    print(f"Max DICE:     {max_dice:.4f}")

evaluate_kmeans_on_training_set_parallel(
    image_paths[:100]
)

ANSWER: Using the Elbow method.
The Elbow method says that k=3 should be the optimal k since it is the point with the biggest distortion slow down (you can see the sharpest change of angle - aka largest change in rate of change). This plots the distortion vs K (sum of suqared distances between data points and their assigned clusters centers). Using KElbowVisualizer, it calculates based in the change in slope and the fit time what is the optimal K and it found that 4 is the optimal K. 

Using the DICE coefficients, 
The DICE coefficient values show that K = 3 is the optimal K since it has the biggest coefficent value. This is in our case the more supperior method of determining the optimal K value since it takes into account the predicted mask and the ground truth mask with pixel to pixel accuracy for each K value. 

ANSWER:

#### 2.1.3. Show (i) the original image, (ii) the segmented image for the optimized value of K. (iii) the ground truth segmentation, available in the csv file corresponding to the image you have chosen.


In [ ]:
# Display the input mask and the ground truth mask
input_mask = cv2.imread(path + 'mask/' + training_image_path + '.png')
input_mask = cv2.cvtColor(input_mask, cv2.COLOR_BGR2GRAY)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(input_mask, cmap='gray')
ax[0].set_title('Provided Ground Truth Mask')
ax[0].axis('off')

ground_truth_segmentation = draw_shapes_from_csv(input_csv, fill=False)
ax[1].imshow(ground_truth_segmentation, cmap='gray')
ax[1].set_title('Generated Ground Truth Segmentation')
ax[1].axis('off')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(input_image)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(color_seg_opt, cmap="gray")
axes[1].set_title(f"K-Means Segmentation with Optimal K (K={optimal_k})")
axes[1].axis("off")

axes[2].imshow(ground_truth_segmentation, cmap="gray")
axes[2].set_title("Ground Truth Segmentation")
axes[2].axis("off")

plt.show()

#### 2.1.4. Do your segmentation results improve if you use a grayscale image compared to an RGB image? <br />Show 3 relevant examples to support your answer.

In [ ]:
label_map_k3, color_seg_k3, centers_k3 = k_means_segmentation(input_image, optimal_k)

In [ ]:
# Need a new function for grayscale images
gray_img = cv2.cvtColor(input_image, cv2.COLOR_RGB2GRAY)

def k_means_segmentation_gray(img, k):
    img_flatten = img.reshape((-1, img.shape[-1])) if len(img.shape) == 3 else img.reshape((-1, 1))
    img_flatten = np.float32(img_flatten)

    # Define criteria and apply k-means
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    _, labels, centers = cv2.kmeans(img_flatten, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

    # Recreate segmented image
    centers = np.uint8(centers)
    segmented_img = centers[labels.flatten()]
    segmented_img = segmented_img.reshape(img.shape)

    return segmented_img

def compare_segmentations(testing_path):
    # Load the original image in RGB and Grayscale
    input_image = cv2.imread(path + 'rgb/' + testing_path + '.png')
    input_image = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(input_image, cv2.COLOR_RGB2GRAY)

    # Segment with the optimal k for both RGB and Grayscale
    label_map_k3, color_seg_k3, centers_k3 = k_means_segmentation(input_image, optimal_k)

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(input_image)
    ax[0].set_title("Original RGB Image")
    ax[0].axis('off')

    ax[1].imshow(color_seg_k3)
    ax[1].set_title("Segmentation (RGB)")
    ax[1].axis('off')

    gray_seg_opt = k_means_segmentation_gray(gray_img, optimal_k)
    ax[2].imshow(gray_seg_opt, cmap='gray')
    ax[2].set_title("Segmentation (Grayscale)")
    ax[2].axis('off')

    plt.show()

testing_paths = ['TCGA-A1-A0SP-DX1_id-5ea4095addda5f8398977ebc_left-7807_top-53960_bottom-54248_right-8089',
        'TCGA-A1-A0SP-DX1_id-5ea4095addda5f8398977ebc_left-11391_top-53955_bottom-54237_right-11688',
        'TCGA-S3-AA15-DX1_id-5ea40a6addda5f839898f24a_left-55998_top-30195_bottom-30473_right-56271']

for testing_path in testing_paths:
    print(f"Testing Path: {testing_path}")
    compare_segmentations(testing_path)


In [ ]:
def compare_rgb_vs_grayscale_kmeans_single(img_path):
    input_image = cv2.imread(path + 'rgb/' + img_path + '.png')
    input_image = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(input_image, cv2.COLOR_RGB2GRAY)
    ground_truth_mask = draw_shapes_from_csv(path + 'csv/' + img_path + '.csv')
    ground_truth_mask = cv2.resize(ground_truth_mask, (input_image.shape[1], input_image.shape[0]))


    # RGB KMeans
    label_map_kmeans, _, _ = k_means_segmentation(input_image, optimal_k)
    dice_rgb = dice_coefficient(label_map_kmeans, ground_truth_mask)

    # Grayscale KMeans
    gray_seg_opt = k_means_segmentation_gray(gray_img, optimal_k)
    dice_gray = dice_coefficient(gray_seg_opt, ground_truth_mask)

    return dice_rgb, dice_gray

# Parallel evaluation across multiple images
def compare_rgb_vs_grayscale_kmeans_parallel(image_paths, n_jobs=-1):
    results = Parallel(n_jobs=n_jobs)(
        delayed(compare_rgb_vs_grayscale_kmeans_single)(img_path)
        for img_path in image_paths
    )

    dice_rgb_all, dice_gray_all = zip(*results)

    print("\n========== RGB vs Grayscale K-Means DICE Summary ==========")
    print(f"Average DICE (RGB): {np.mean(dice_rgb_all):.4f}")
    print(f"Average DICE (Grayscale): {np.mean(dice_gray_all):.4f}")
    print(f"Max DICE (RGB): {np.max(dice_rgb_all):.4f}")
    print(f"Max DICE (Grayscale): {np.max(dice_gray_all):.4f}")
    print(f"Min DICE (RGB): {np.min(dice_rgb_all):.4f}")
    print(f"Min DICE (Grayscale): {np.min(dice_gray_all):.4f}")

# Compare RGB vs Grayscale KMeans
compare_rgb_vs_grayscale_kmeans_parallel(image_paths[:100])

In [ ]:
gray_seg_opt = k_means_segmentation_gray(gray_img, optimal_k)

Using grayscale, we can see that it is much quicker to run than using RGB and we obtain slightly better results when comparing the DICE scores for both. We can also see that in the image, we are getting verry similar results, arguably RGB produces a better segmented image, but with the time that we save by using grayscale, it is verry much worth it to use it instead of RGB. 

#### 2.1.5. You have studied a number of other unsupervised segmentation methods in class. <br /> Pick one and compare the results to the K-means results (for the optimal K found above). Run your code on the RGB image.


In [ ]:
from sklearn.cluster import MeanShift, estimate_bandwidth
#https://www.datacamp.com/tutorial/mean-shift-clustering
# Function to perform Mean Shift Segmentation
def mean_shift_segmentation(img, bandwidth):
    h, w, ch = img.shape
    img_flatten = img.reshape(h * w, ch)  # Flatten the image to a 2D array of pixels
    mean_shift = MeanShift(bandwidth=bandwidth, bin_seeding=True)
    labels = mean_shift.fit_predict(img_flatten)
    segmented_img = mean_shift.cluster_centers_[labels].reshape(h, w, ch).astype(np.uint8)
    return segmented_img


bandwidth = estimate_bandwidth(input_image.reshape(-1, 3), quantile=0.09, n_samples=500)
seg_img_mean_shift_30 = mean_shift_segmentation(input_image, bandwidth)


fig, ax = plt.subplots(1, 3, figsize=(20, 5))
ax[0].imshow(input_image)
ax[0].set_title('Original Image')
ax[0].axis('off')

ax[1].imshow(color_seg_opt)
ax[1].set_title(f'Segmentation with Optimal K (K={optimal_k}) - KMeans')
ax[1].axis('off')

ax[2].imshow(seg_img_mean_shift_30)
ax[2].set_title(f'Segmentation with Mean Shift (Bandwidth={bandwidth})')
ax[2].axis('off')
plt.show()

def rgb_to_label_map(seg_img):
    reshaped = seg_img.reshape(-1, 3)
    unique_colors, labels = np.unique(reshaped, axis=0, return_inverse=True)
    return labels.reshape(seg_img.shape[:2])

# Compare K-Means and Mean Shift for same input image
def compare_kmeans_vs_meanshift_single(img_path, max_k=10):
    input_image = cv2.imread(path + 'rgb/' + img_path + '.png')
    input_image = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    ground_truth_mask = draw_shapes_from_csv(path + 'csv/' + img_path + '.csv')
    ground_truth_mask = cv2.resize(ground_truth_mask, (input_image.shape[1], input_image.shape[0]))

    # KMeans
    optimal_k, best_dice_kmeans = find_optimal_k_dice(input_image, ground_truth_mask, False, max_k)
    label_map_kmeans, _, _ = k_means_segmentation(input_image, optimal_k)
    dice_kmeans = dice_coefficient(label_map_kmeans.astype(np.uint8), ground_truth_mask.astype(np.uint8))

    #Mean Shift
    bandwidth = estimate_bandwidth(input_image.reshape(-1, 3), quantile=0.09, n_samples=500)
    seg_img_meanshift = mean_shift_segmentation(input_image, bandwidth)
    label_map_meanshift = rgb_to_label_map(seg_img_meanshift)
    dice_meanshift = dice_coefficient(label_map_meanshift.astype(np.uint8), ground_truth_mask.astype(np.uint8))

    return optimal_k, bandwidth, dice_kmeans, dice_meanshift

# Parallel evaluation across multiple images
def compare_kmeans_vs_meanshift_parallel(image_paths, max_k=10, n_jobs=-1):
    results = Parallel(n_jobs=n_jobs)(
        delayed(compare_kmeans_vs_meanshift_single)(img_path, max_k)
        for img_path in image_paths
    )

    optimal_ks, bandwidths, dice_kmeans_all, dice_meanshift_all = zip(*results)

    print("\n========== K-Means vs Mean Shift DICE Summary ==========")
    print(f"Average DICE (K-Means): {np.mean(dice_kmeans_all):.4f}")
    print(f"Average DICE (Mean Shift): {np.mean(dice_meanshift_all):.4f}")
    print(f"Max DICE (K-Means): {np.max(dice_kmeans_all):.4f}")
    print(f"Max DICE (Mean Shift): {np.max(dice_meanshift_all):.4f}")
    print(f"Min DICE (K-Means): {np.min(dice_kmeans_all):.4f}")
    print(f"Min DICE (Mean Shift): {np.min(dice_meanshift_all):.4f}")


compare_kmeans_vs_meanshift_parallel(image_paths[:100])



In [ ]:
bandwidth = estimate_bandwidth(input_image.reshape(-1, 3), quantile=0.09, n_samples=500)
seg_img_mean_shift_30 = mean_shift_segmentation(input_image, bandwidth)

### 2.2. Segmentation - Supervised

#### 2.2.1. Describe the supervised approach you are using.

In [ ]:
import pandas as pd
import joblib
from skimage.feature import local_binary_pattern
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, normalized_mutual_info_score, adjusted_rand_score
from scipy.ndimage import binary_opening, binary_closing

In [ ]:
# Define folder paths
rgb_folder = path+"rgb/"
mask_folder = path+"mask/"
csv_folder = path+"csv/"
## split_file = path+"splits/validate_subjects.csv"
split_file = path+"splits/train_subjects.csv"

# Get the list of subjects from the training set
file_content = pd.read_csv(split_file, header=None)
prefixes = set(file_content[0].tolist())

In [ ]:
# Load images from a folder
def load_images(folder, color_mode=cv2.IMREAD_COLOR):
    image_dict = {}
    for filename in os.listdir(folder):
        subjectname = filename.split('_')[0] + '_id'
        if subjectname in prefixes:
            image = cv2.imread(folder+"/"+filename)
            if(image is not None):
                image_dict[filename] = image
    return image_dict

In [ ]:
# Load csv annotations
def load_csv(folder):
    csv_dict = {}
    for filename in os.listdir(folder):
        subjectname = filename.split('_')[0] + '_id'
        if subjectname in prefixes:
            df = pd.read_csv(folder+"/"+filename)
            csv_dict[filename] = df
    return csv_dict

In [ ]:
# Generate a global label map from all main_classification values within all csv files stored in a dictionary
def generate_label_map(csv_annotations):
    all_labels = set()
    for df in csv_annotations.values():
        all_labels.update(df["raw_classification"].unique())
    global_label_map = {label: idx for idx, label in enumerate(sorted(all_labels))}
    return global_label_map

In [ ]:
# Convert annotation labels to numerical class IDs
def encode_labels(df, label_map):
    df["class_id"] = df["raw_classification"].map(label_map)
    return df, label_map

In [ ]:
# Extract features from image
def extract_features(image, hist_bins=6):
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Compute LBP & Sobel
    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

    mean_color = cv2.mean(image)[:3]
        
    feature_vector = np.dstack((gray, lbp, sobel_x, sobel_y)).reshape(-1, 4)
    return np.hstack((feature_vector, np.array(mean_color).reshape(1, -1).repeat(len(feature_vector), axis=0)))

In [ ]:
# Load the training images
rgb_images_train = load_images(rgb_folder)
mask_images_train = load_images(mask_folder, color_mode=cv2.IMREAD_UNCHANGED)
csv_annotations = load_csv(csv_folder) # Tuple of (filename, DataFrame)
print(len(rgb_images_train))
print(len(mask_images_train))
print(len(csv_annotations))

x, y = [], []

# Define the percentage of images to use within the entire training dataset
sample_fraction = 0.12

# Get a random subset of image filenames
selected_filenames = np.random.choice(list(rgb_images_train.keys()), int(sample_fraction * len(rgb_images_train)), replace=False)

# Filter datasets to only keep selected images
rgb_images = {k: rgb_images_train[k] for k in selected_filenames}
mask_images = {k: mask_images_train[k] for k in selected_filenames if k in mask_images_train}

print(len(rgb_images))
print(len(mask_images))

In [ ]:
# Generate the label map for all the different raw_classification values across the csv data
training_label_map = generate_label_map(csv_annotations)
print(training_label_map)

In [ ]:
# Iterate through the selected images and their corresponding masks
for filename, rgb_img in rgb_images.items():
    if filename in mask_images:
        mask = mask_images[filename]

        # Check if the dimensions of rgb_img and mask match and resize if they don't
        if rgb_img.shape[:2] != mask.shape[:2]:
            mask = cv2.resize(mask, (rgb_img.shape[1], rgb_img.shape[0]), interpolation=cv2.INTER_NEAREST)

        csv_name = filename[:-3]+"csv"
        df = csv_annotations[csv_name]
        df, label_map = encode_labels(df, training_label_map)

        # Extract class labels from the first channel of the mask and instance labels as a product of the second and third channels
        class_labels = mask[:, :, 0]
        instance_labels = mask[:, :, 1] * mask[:, :, 2]

        features = extract_features(rgb_img)
        labels = np.zeros_like(class_labels)

        # Assign class labels bbased on the bounding box annotations
        for _, row in df.iterrows():
            xmin, ymin, xmax, ymax = row["xmin"], row["ymin"], row["xmax"], row["ymax"]
            mask_region = class_labels[ymin:ymax, xmin:xmax]
            instance_region = instance_labels[ymin:ymax, xmin:xmax]

            #Assign labels where instance mask is present
            valid_indices = instance_region>0
            labels[ymin:ymax, xmin:xmax][valid_indices] = row["class_id"]
        
        x.append(features)
        y.append(labels.flatten())
x = np.vstack(x)
y = np.hstack(y)

# Remove background and FOV area (assuming 0 is background and FOV pixels)
valid_indices = y > 0
x = x[valid_indices]
y = y[valid_indices]
print(len(x))
print(len(y))

In [ ]:
# Data scaling and fitting
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# Train RF model
rf = RandomForestClassifier(n_estimators=100, max_depth=38, random_state=42, n_jobs=-1)
rf.fit(x_train, y_train)

#### 2.2.2. Report the performance of your supervised segmentor on the training images.

In [ ]:
# Evaluate model using the batch of training images that was used to report the performance of the supervised segmentor
predictions = rf.predict(x_test)
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, average='weighted', zero_division=0)
recall = recall_score(y_test, predictions, average='weighted', zero_division=0)

print(f"Random Forest model accuracy: {accuracy:.4f}")
print(f"Trained model precision on training images: {precision:.4f}")
print(f"Trained model recall on training images: {recall:.4f}")


The model has an accuracy of 80.19% and a precision of 78.45% when generating segmentation masks for the images within the training dataset.

In [ ]:
# Method to apply the trained Random Forest model for segmentation
def segment_image(image):
    features = extract_features(image)
    features = scaler.transform(features)
    predictions = rf.predict(features)
    return predictions.reshape(image.shape[:2])

def post_process_segmentation(segmentation):
    processed = np.zeros_like(segmentation)
    unique_classes = np.unique(segmentation)

    for cls in unique_classes:
        if cls == 0:  # Skip background
            continue
        mask = segmentation == cls
        mask = binary_opening(mask)  # Remove small noise for this class
        mask = binary_closing(mask)  # Fill holes for this class
        processed[mask] = cls  # Assign back the class label
    
    return processed

input_image = cv2.imread(path + 'rgb/TCGA-A1-A0SP-DX1_id-5ea4095addda5f8398977ebc_left-7807_top-53960_bottom-54248_right-8089.png', cv2.IMREAD_COLOR)
ground_truth_mask = cv2.imread(path + 'mask/TCGA-A1-A0SP-DX1_id-5ea4095addda5f8398977ebc_left-7807_top-53960_bottom-54248_right-8089.png', cv2.IMREAD_GRAYSCALE)
segmentation_mask = segment_image(input_image)
processed_mask = post_process_segmentation(segmentation_mask)
print(np.unique(segmentation_mask))

plt.figure(figsize=(20,20))
plt.subplot(1,4,1)
plt.imshow(input_image)
plt.title("Original image")
plt.xticks([]), plt.yticks([])
plt.subplot(1,4,2)
plt.imshow(segmentation_mask)
plt.title("Segmentation mask")
plt.xticks([]), plt.yticks([])
plt.subplot(1,4,3)
plt.imshow(processed_mask)
plt.title("Post-processed segmentation mask")
plt.xticks([]), plt.yticks([])
plt.subplot(1,4,4)
plt.imshow(ground_truth_mask, cmap="gray")
plt.title("Ground truth mask")
plt.xticks([]), plt.yticks([])
plt.show()

In [ ]:
# Compute DICE coefficient between two binary masks
def dice_coefficient(pred_mask, true_mask):
    # Check if the dimensions of pred_mask and true_mask match and resize if they don't
    if true_mask.shape[:2] != pred_mask.shape[:2]:
                true_mask = cv2.resize(true_mask, (pred_mask.shape[1], pred_mask.shape[0]))
    
    pred_mask = (pred_mask > 0).astype(np.uint8)
    true_mask = (true_mask > 0).astype(np.uint8)
    
    intersection = np.sum(pred_mask * true_mask)
    total = np.sum(pred_mask) + np.sum(true_mask)
    
    return (2.0 * intersection) / total if total > 0 else 0.0


print("DICE score for segmentation produced by trained supervised segmentation model:", dice_coefficient(segmentation_mask, ground_truth_mask))

#### 2.2.3. Report the performance of your supervised segmentor on the validation images (images for the subjects in val_subjects.csv).

In [ ]:
# Read the subject ids from validate_subjects.csv
split_file = path+"splits/validate_subjects.csv"
file_content = pd.read_csv(split_file, header=None)
prefixes = set(file_content[0].tolist())

# Load the validation dataset images
rgb_images_test = load_images(rgb_folder)
mask_images_test = load_images(mask_folder)
csv_annotations_test = load_csv(csv_folder)

# Define the percentage of images to use within the entire validation dataset
sample_fraction = 0.2

# Get a random subset of image filenames
selected_filenames = np.random.choice(list(rgb_images_test.keys()), int(sample_fraction * len(rgb_images_test)), replace=False)

# Filter datasets to only keep selected images
rgb_images = {k: rgb_images_test[k] for k in selected_filenames}
mask_images = {k: mask_images_test[k] for k in selected_filenames if k in mask_images_test}

# Generate the label map for all the different raw_classification values across the csv data
test_label_map = generate_label_map(csv_annotations)

In [ ]:
x_test, y_test = [], []
for filename, rgb_img in rgb_images.items():
    if filename in mask_images:
        mask = mask_images[filename]

        # Check if the dimensions of rgb_img and mask match and resize if they don't
        if rgb_img.shape[:2] != mask.shape[:2]:
            mask = cv2.resize(mask, (rgb_img.shape[1], rgb_img.shape[0]), interpolation=cv2.INTER_NEAREST)

        csv_name = filename[:-3]+"csv"
        df = csv_annotations_test[csv_name]
        df, label_map = encode_labels(df, test_label_map)

        # Extract class labels from the first channel of the mask and instance labels as a product of the second and third channels
        class_labels = mask[:, :, 0]
        instance_labels = mask[:, :, 1] * mask[:, :, 2]

        features = extract_features(rgb_img)
        labels = np.zeros_like(class_labels)

        # Assign class labels bbased on the bounding box annotations
        for _, row in df.iterrows():
            xmin, ymin, xmax, ymax = row["xmin"], row["ymin"], row["xmax"], row["ymax"]
            mask_region = class_labels[ymin:ymax, xmin:xmax]
            instance_region = instance_labels[ymin:ymax, xmin:xmax]

            #Assign labels where instance mask is present
            valid_indices = instance_region>0
            labels[ymin:ymax, xmin:xmax][valid_indices] = row["class_id"]
        
        x_test.append(features)
        y_test.append(labels.flatten())
x_test = np.vstack(x_test)
y_test = np.hstack(y_test)

# Remove background pixels (y_test > 0) to match training setup
valid_indices = y_test > 0
x_test = x_test[valid_indices]
y_test = y_test[valid_indices]


In [ ]:
# Standardize test features using the same scaler from training
x_test = scaler.transform(x_test)

# Predict segmentation labels for the test dataset
test_predictions = rf.predict(x_test)

# Evaluate model on test dataset
test_accuracy = accuracy_score(y_test, test_predictions)
test_precision = precision_score(y_test, test_predictions, average='weighted', zero_division=0)
test_recall = recall_score(y_test, test_predictions, average='weighted', zero_division=0)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")

The model has an accuracy of 42.95% and a precision of 37.16% when it is generating the segmentation masks for images within the validation dataset. While this is significantly lower than the accuracy and precision when working with the training dataset, training the model on a larger portion of the entire training dataset would result in better performance. Hardware limitations were a huge limiting factor in this regard.

In [ ]:
# Get a random file from the test dataset and generate the segmentation mask for it
filename = np.random.choice(list(rgb_images.keys()))
input_image = cv2.imread(rgb_folder + filename, cv2.IMREAD_COLOR)
ground_truth_mask = cv2.imread(mask_folder + filename, cv2.IMREAD_GRAYSCALE)
segmentation_mask = segment_image(input_image)
processed_mask = post_process_segmentation(segmentation_mask)

plt.figure(figsize=(20,20))
plt.subplot(1,4,1)
plt.imshow(input_image)
plt.title("Original image")
plt.xticks([]), plt.yticks([])
plt.subplot(1,4,2)
plt.imshow(segmentation_mask)
plt.title("Segmentation mask")
plt.xticks([]), plt.yticks([])
plt.subplot(1,4,3)
plt.imshow(processed_mask)
plt.title("Post-processed segmentation mask")
plt.xticks([]), plt.yticks([])
plt.subplot(1,4,4)
plt.imshow(ground_truth_mask, cmap="gray")
plt.title("Ground truth mask")
plt.xticks([]), plt.yticks([])
plt.show()


In [ ]:
# Print DICE score:
print("DICE score for segmentation produced on validation image by trained supervised segmentation model:", dice_coefficient(segmentation_mask, ground_truth_mask))

#### 2.2.4. Select a few samples (at least 3) from the validation images and compare the performance of the supervised and unsupervised methods (i) qualitatively and (ii) quantitatively. <br />Discuss which method (supervised or unsupervised) has better performance.

In [ ]:
# Method to load the sample data from an array contianing the filenames for the rgb images
def load_sample_data(samples):
    rgb_images = []
    mask_images = []
    csv_dataframes = []

    for filename in samples:
        # Build full paths
        rgb_path = rgb_folder+filename
        mask_path = mask_folder+filename
        csv_filename = filename.replace(".png", ".csv")
        csv_path = csv_folder+csv_filename

        # Load images and CSV
        rgb_image = cv2.imread(rgb_path, cv2.IMREAD_COLOR)
        mask_image = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        csv_df = pd.read_csv(csv_path)

        # Append to respective lists
        rgb_images.append(rgb_image)
        mask_images.append(mask_image)
        csv_dataframes.append(csv_df)

    return rgb_images, mask_images, csv_dataframes

In [ ]:
# Method to generate supervised segmentation masks for all the images within an array
def generate_supervised_segmentations(imgs):
    segmentation_masks = []
    processed_masks = []
    for i in range(len(imgs)):
        rgb = imgs[i]
        segmentation_mask = segment_image(rgb)
        processed_mask = post_process_segmentation(segmentation_mask)
        segmentation_masks.append(segmentation_mask)
        processed_masks.append(processed_mask)
    return segmentation_masks, processed_masks

In [ ]:
# Method to generate unsupervised segmentation masks for all the images within an array
def generate_unsupervised_segmentations(imgs):
    segmentation_masks = []
    for i in range(len(imgs)):
        img = imgs[i]
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_flat = img.reshape((-1, 3))
        h, w, ch = img.shape
        kmeans = KMeans(n_clusters=3, random_state=0, n_init=10)
        labels = kmeans.fit_predict(img_flat)
        segmented_img = kmeans.cluster_centers_[labels].reshape(h, w, 3).astype(np.uint8)
        segmentation_masks.append(segmented_img)
    return segmentation_masks

In [ ]:
# Method to display all segmentation, rgb, mask and post processed images from arrays in a row
def display_segmentation_images(rgb_images, mask_images, segmentation_images, post_processed_images):
    for i in range(len(rgb_images)):
        plt.figure(figsize=(20,20))
        plt.subplot(1,4,1)
        plt.imshow(rgb_images[i])
        plt.title("Original image")
        plt.xticks([]), plt.yticks([])
        plt.subplot(1,4,2)
        plt.imshow(segmentation_images[i])
        plt.title("Segmentation mask")
        plt.xticks([]), plt.yticks([])
        plt.subplot(1,4,3)
        plt.imshow(post_processed_images[i])
        plt.title("Post-processed segmentation mask")
        plt.xticks([]), plt.yticks([])
        plt.subplot(1,4,4)
        plt.imshow(mask_images[i], cmap="gray")
        plt.title("Ground truth mask")
        plt.xticks([]), plt.yticks([])
        plt.show()


In [ ]:
# Method to display all unsupervised segmentation, rgb and mask images from arrays in a row
def display_unsupervised_segmentation_images(rgb_images, mask_images, segmentation_images):
    for i in range(len(rgb_images)):
        plt.figure(figsize=(20,20))
        plt.subplot(1,3,1)
        plt.imshow(rgb_images[i])
        plt.title("Original image")
        plt.xticks([]), plt.yticks([])
        plt.subplot(1,3,2)
        plt.imshow(segmentation_images[i])
        plt.title("Segmentation mask")
        plt.xticks([]), plt.yticks([])
        plt.subplot(1,3,3)
        plt.imshow(mask_images[i], cmap="gray")
        plt.title("Ground truth mask")
        plt.xticks([]), plt.yticks([])
        plt.show()

##### i) Supervised:

In [ ]:
samples = []
samples.append('TCGA-D8-A27F-DX1_id-5ea409e2ddda5f83989878b6_left-100834_top-8259_bottom-8535_right-101117.png')
samples.append('TCGA-AN-A0AT-DX1_id-5ea40a18ddda5f839898a4f0_left-34219_top-38810_bottom-39110_right-34498.png')
samples.append('TCGA-GI-A2C9-DX1_id-5ea40a03ddda5f83989894a4_left-23944_top-12608_bottom-12869_right-24227.png')

sample_rgb, sample_mask, sample_csv = load_sample_data(samples)
segmentation_masks, processed_masks = generate_supervised_segmentations(sample_rgb)
display_segmentation_images(sample_rgb, sample_mask, segmentation_masks, processed_masks)

We can perform a quantitative comparison of a supervised segmentation model with unsupervised segmentation by computing the DICE scores for the segmentation masks generated by both methods of segmentation. DICE coefficient measures the spatial overlap between predicted and true masks. It directly compares pixel-wise agreement for each class which causes it to be a well-suited evaluation metric for semantic segmentation.

In [ ]:
# Method to print the DICE score for each predicted segmentation mask and its corresponding ground truth mask
# Displays the average DICE score as well

def evaluate_segmentation(pred_masks, true_masks):
    dice_score = 0
    for i in range(len(pred_masks)):
        pred_mask = pred_masks[i]
        true_mask = true_masks[i]
        # Convert predicted image to grayscale if it's RGB
        if pred_mask.ndim == 3 and pred_mask.shape[2] == 3:
            pred_mask = cv2.cvtColor(pred_mask, cv2.COLOR_BGR2GRAY)
        # Check if the dimensions of pred_mask and true_mask match and resize if they don't
        if true_mask.shape[:2] != pred_mask.shape[:2]:
            true_mask = cv2.resize(true_mask, (pred_mask.shape[1], pred_mask.shape[0]))
        
        score = dice_coefficient(pred_mask, true_mask)
        dice_score+=score
        print("DICE score:", score)

    avg_score = dice_score / len(segmentation_masks)
    print("Average DICE score for segmentation masks generated using validation set of images:", avg_score)

In [ ]:
evaluate_segmentation(segmentation_masks, sample_mask)

##### ii) Unsupervised:

In [ ]:
unsupervised_segmentation_masks = generate_unsupervised_segmentations(sample_rgb)
display_unsupervised_segmentation_images(sample_rgb, sample_mask, unsupervised_segmentation_masks)

In [ ]:
evaluate_segmentation(unsupervised_segmentation_masks, sample_mask,)

- Quantitative Analysis:
    For supervised segmentation, the average DICE score is around 0.9288 and for unsupervised segmentation it is around 0.8986. This indicates that the quality of the segmentation mask generated by the supervised segmentation method is slightly better than the unsupervised segmentation implementation.

- Qualitative Analysis:
    The supervised segmentation model was trained on a tenth of the total training dataset size due to hardware limitations and time constraints. However, the results obtained with this limited training of the model were satisfactory, obtaining a 48% accuracy when predicting the class labels of the images in the test dataset. Looking at the images, however, it is clear that the unsupervised, k-means segmentation offers better outlining of the different nuclei and more well-defined segmentation. The supervised segmentation seems to be trying to preserve a lot of the granular details resulting in a less defined segmentation mask. 

All things considered, the supervised segmentation method offers better performance in terms of DICE scores. However, qualitatively, the output of the unsupervised segmentation mask seems to provide better distinction of nuclei classes. If hardware limitations were not a factor and it was possible to train the Random Forest model on a larger dataset and include more image features to be extracted from each dataset image such as color histograms, it is very likely that the Random Forest model could outperform unsupervised segmentation.

#### 2.2.5. For a given subject, which nuclei group can be segmented easily? Discuss by showing some sample visualizations.


In [ ]:
subject_id = 'TCGA-AN-A0AL-DX1_id'

# Load rgb, mask and csv data from all scans belonging to a subject
def load_data_subject(subject):
    rgb = []
    mask = []
    csv = []
    for filename in os.listdir(rgb_folder):
        subjectname = filename.split('_')[0] + '_id'
        if subjectname == subject:
            image = cv2.imread(rgb_folder+filename, cv2.IMREAD_COLOR)
            if(image is not None):
                rgb.append(image)
    for filename in os.listdir(mask_folder):
        subjectname = filename.split('_')[0] + '_id'
        if subjectname == subject:
            image = cv2.imread(mask_folder+filename, cv2.IMREAD_COLOR)
            if(image is not None):
                mask.append(image)
    for filename in os.listdir(csv_folder):
        subjectname = filename.split('_')[0] + '_id'
        if subjectname == subject:
            df = pd.read_csv(csv_folder+filename)
            csv.append(df)
    return rgb, mask, csv

subject_rgb, subject_mask, subject_csv = load_data_subject(subject_id)
# Use our trained RF model to generate segmentation masks for all the images
segmentation_mask_subject, processed_mask_subject = generate_supervised_segmentations(subject_rgb)

In [ ]:
# Method to generate a mask image out of the csv file following the same label values used when training the RF model
def csv_to_mask(df, image_shape, label_map):
    mask = np.zeros(image_shape[:2], dtype=np.uint8)
    for _, row in df.iterrows():
        label_name = row['raw_classification']
        label_value = label_map[label_name]
        try:
            x_coords = list(map(int, str(row['coords_x']).split(',')))
            y_coords = list(map(int, str(row['coords_y']).split(',')))
        except Exception as e:
            print(f"Error parsing coordinates: {e}")
            continue
        
        if len(x_coords) != len(y_coords):
            print(f"Skipping row due to mismatched coordinates: {x_coords} vs {y_coords}")
            continue
        points = np.array([list(zip(x_coords, y_coords))], dtype=np.int32)
        cv2.fillPoly(mask, points, color=label_value)

    return mask


In [ ]:
from collections import defaultdict

subject_label_map = generate_label_map(csv_annotations)
generated_mask = csv_to_mask(subject_csv[0], subject_rgb[0].shape, subject_label_map)

# Method to quanitfy the segmentation of each class of nuclei across the generated segmentation masks
class_wise_scores = defaultdict(list)
for i in range(len(subject_csv)):
    generated_mask = csv_to_mask(subject_csv[i], subject_rgb[i].shape, subject_label_map)
    seg_mask = segmentation_mask_subject[i]

    common_mask = (generated_mask > 0)

    seg_mask_flat = seg_mask[common_mask].flatten()
    generated_mask_flat = generated_mask[common_mask].flatten()

    for label in np.unique(generated_mask_flat):
        label_mask = (generated_mask_flat == label)
        score = accuracy_score(generated_mask_flat[label_mask], seg_mask_flat[label_mask])
        class_wise_scores[label].append(score)

# Print the average scores for the accuracy based on the label of each nucleus
average_scores = {label: np.mean(scores) for label, scores in class_wise_scores.items()}
print(average_scores)
# Print the label map to see which nuclei each index corresponds to within the label map
print(subject_label_map)


In [ ]:
# Visualize the correctly identified nuclei
top_labels = sorted(average_scores, key=average_scores.get, reverse=True)[:3]

def visualize_predictions(samples):
    for mask in samples:
        plt.figure(figsize=(12,4))
        plt.imshow(mask, cmap = "tab20")
        plt.axis('off')
        plt.show()

for label in top_labels:
    matching_samples = []
    for mask, csv in zip(segmentation_mask_subject, subject_csv):
        gt_mask = csv_to_mask(csv, mask.shape, subject_label_map)
        if np.any(gt_mask == label):
            matching_samples.append(mask)
    visualize_predictions(matching_samples)

The figures displayed above show the correctly identified nuclei types in different segmentation masks. Some masks appear twice with different colors and different structures due to different types of nuclei being identified correctly within the same mask. For this particular subject, TCGA-AN-A0AL-DX1_id, we generated the average scores before displaying the correctly identified nuclei. Using the csv file data, we assigned label mappings to different nuclei types and simply compared the label mappings in the generated segmentation mask with that of the mask generated from the csv data. We assigned an accuracy score based on how many pixels had the correct type assigned within the segmentation mask in comparison to the mask generated from the csv data. Looking at the data, it seems like tumors, fibroblasts, lymphocytes and unlabeled cells were the most easily segmented. Tumors were significantly more easily segmented than the other nuclei types.

#### 2.2.6. Why do you think the nuclei you mentioned in the previous question are easy to segment?

Analyzing the csv for the subject TCGA-AN-A0AL-DX1_id,  it seems like the nuclei types that were most easily segmented also appeared the most often within the images. It is also highly likely that this trend is reflected in the training dataset of the model, causing it to be skewed towards prioritizing detection of these nuclei more than the others. Additionally, the model was trained on a small subset of the training dataset. It is possible that the results might be different were the model trained on a different subset of the training dataset. Another reasoning may be that the weight of tumors was more prioritized by the model, or the feature extraction step suited tumor detection better than other types of cells.

## 3. Detecting and Counting Tumors

### 3.1. Preparing the data

#### 3.1.1. Convert input images and masks to grayscale.

In [ ]:
# Given the image name, convert the image and corresponding mask to grayscale

def convert_img_and_mask_to_grayscale(image_name):
    # Load the BGR images
    img = cv2.imread(path + 'rgb/' + image_name + '.png')
    mask = cv2.imread(path + 'mask/' + image_name + '.png')
    
    # Convert to grayscale
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask_gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
    return img_gray, mask_gray

input_image_name = training_image_path
img_gray, mask_gray = convert_img_and_mask_to_grayscale(input_image_name)

img = cv2.imread(path + 'rgb/' + input_image_name + '.png')
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

mask = cv2.imread(path + 'mask/' + input_image_name + '.png')
mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(7, 7))
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title('Original Image')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(mask)
plt.title('Mask Image')
plt.axis('off')
plt.show()

# Display the images
plt.figure(figsize=(7, 7))
plt.subplot(1, 2, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('Grayscale Image')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(mask_gray, cmap='gray')
plt.title('Grayscale Mask')
plt.axis('off')
plt.show()

#### 3.1.2. Multiply the mask and the original image to obtain the regions of interest.


In [ ]:
# Cuts the regions of interest out of the image using the mask
def get_interest_regions(img_gray, mask_gray):
    
    # Slice both images to the same size
    if img_gray.shape != mask_gray.shape:
        # print("Image and mask sizes do not match!")
        # Slice the bigger image to match the mask size
        m_h, m_w = mask_gray.shape
        i_h, i_w = img_gray.shape
        
        # Compare the dimensions and slice the larger one to match the smaller one
        if m_h > i_h:
            mask_gray = mask_gray[:i_h, :m_w]
        else:
            img_gray = img_gray[:m_h, :i_w]
        
        if m_w > i_w:
            mask_gray = mask_gray[:m_h, :i_w]
        else:
            img_gray = img_gray[:i_h, :m_w]
    
    # Make mask pixels that are 0 into np.max(mask_gray)
    # This is to ensure that the background is white (255) and the foreground is black (0)
    mask_gray[mask_gray == 0] = np.max(mask_gray)
    
    # Invert the mask to get the background
    mask_gray = cv2.bitwise_not(mask_gray)
    
    # Binarize the mask
    _, mask_binary = cv2.threshold(mask_gray, np.min(mask_gray), 255, cv2.THRESH_BINARY)
    
    # Apply the mask to the image
    img_interest_regions = cv2.bitwise_and(img_gray, mask_binary)
    
    # plt.figure(figsize=(7, 7))
    # plt.subplot(1, 2, 1)
    # plt.imshow(mask_binary, cmap='gray')
    # plt.title('Inverted and Binarized Mask')
    # plt.axis('off')
    # plt.subplot(1, 2, 2)
    # plt.imshow(img_interest_regions, cmap='gray')
    # plt.title('Image with ROI')
    # plt.axis('off')
    # plt.show()
    
    return img_interest_regions 

img_interest_regions = get_interest_regions(img_gray, mask_gray)

# Display the image with the background removed
plt.figure(figsize=(5, 5))
plt.imshow(img_interest_regions, cmap='gray')
plt.title('Image with ROI')
plt.axis('off')
plt.show()

#### 3.1.3. Remove the background noise in the resultant image.

In [ ]:
# Remove background noise from the image
def remove_background_noise(img_interest_regions):
    
    img_interest_regions_copy = img_interest_regions.copy()
    
    # XXX - Note: Currently this step is technically being done in the get_interest_regions function - XXX
    # The following code is commented out because it is not needed for the current implementation but could be used to further remove background noise
    
    # # Get the mean and std of the non-zero pixels in the image
    # non_zero_mean = np.mean(img_interest_regions_copy[img_interest_regions_copy != 0])
    # non_zero_std = np.std(img_interest_regions_copy[img_interest_regions_copy != 0])
    
    # # There is still some background noise in the image, so we need to remove it
    # img_interest_regions_copy[img_interest_regions > non_zero_mean + 2*non_zero_std] = 0

    return img_interest_regions_copy

# img_interest_regions = get_interest_regions(img_gray, mask_gray)
prepared_image = remove_background_noise(img_interest_regions)

# Display the image
plt.figure(figsize=(7, 7))
plt.subplot(1, 2, 1)
plt.imshow(img_interest_regions, cmap='gray')
plt.title('Image with ROI')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(prepared_image, cmap='gray')
plt.title('Image with ROI and\nBackground Noise Removed')
plt.axis('off')
plt.show()

#### 3.1.4. Extract features of the resultant image that has different pathological structures (blobs).

In [ ]:
# Extract blobs from the resultant image
def extract_blobs(prepared_image, csv_data):
    img_features = prepared_image.copy()
    
    # For each row in the CSV file, get the coordinates of the points 
    bounding_boxes = []  # List to store bounding boxes
    coords_x = []
    coords_y = []
    for _, row in csv_data.iterrows():
        xmin, ymin, xmax, ymax = row['xmin'], row['ymin'], row['xmax'], row['ymax']
        coords_x.append(ast.literal_eval(row['coords_x']))
        coords_y.append(ast.literal_eval(row['coords_y']))
        bounding_boxes.append(cv2.boundingRect(np.array([[xmin, ymin], [xmax, ymax]])))
    
    # Display the bounding boxes
    blobs = []
    blob_coords = []
    skip_indexes = []
    for i, (x, y, w, h) in enumerate(bounding_boxes):
        # If smaller than 5 x 5, skip it
        if w < 5 or h < 5:
            skip_indexes.append(i)
            continue
        
        image = img_features[y:y + h, x:x + w]
        blobs.append(image)
        
        # Display the cut image
        # plt.figure(figsize=(5, 5))
        # plt.imshow(image, cmap='gray')
        # plt.title("Cut Blob - Coords | x: {}, y: {}, w: {}, h: {}".format(x, y, w, h))
        # plt.axis('off')
        # plt.show()
        xmin, ymin, xmax, ymax = x, y, x + w, y + h # Bounding box coordinates
        # Display the bounding boxes
        # print(f"xmin: {xmin}, ymin: {ymin}, xmax: {xmax}, ymax: {ymax}")
        blob_coords.append((xmin, ymin, xmax, ymax, coords_x[i], coords_y[i]))
    
    return blobs, blob_coords, skip_indexes

csv_data = pd.read_csv(path + 'csv/' + training_image_path + '.csv')
blobs_gray, blob_coords, skipped_blobs = extract_blobs(prepared_image, csv_data)

print(f"Skipped blobs: {skipped_blobs}")

hs = []
ws = []
for blob in blobs_gray:
    h, w = blob.shape
    hs.append(h)
    ws.append(w)
print(f"Average height: {int(np.mean(hs))}, Average width: {int(np.mean(ws))}")
print(f"Max height: {np.max(hs)}, Max width: {np.max(ws)}")
print(f"Min height: {np.min(hs)}, Min width: {np.min(ws)}")

print(f"Number of blobs: {len(blobs_gray)}")
print(f"Blob shapes: {[blob.shape for blob in blobs_gray]}")
print(f"Blob coordinates: {blob_coords}")

example_blob = blobs_gray[1]

In [ ]:
# Function to extract HOG features from the blobs
def get_hog_features(images):
    # Parameters for HOG
    cell_size = (8, 8)
    block_size = (2, 2)
    nbins = 9
    block_norm = 'L2'

    hog_features = []
    for img in images:
        # Resize the image to a fixed size
        resized_img = cv2.resize(img, (64, 64))
        features = hog(resized_img, orientations=nbins, pixels_per_cell=cell_size, cells_per_block=block_size, block_norm=block_norm, visualize=False, feature_vector=True)
        hog_features.append(features)
    return np.array(hog_features)

# Extract HOG features from the grayscale image
print("Extracting HOG features...")
hog_features = get_hog_features(blobs_gray)
print("Extracted HOG features shape:", hog_features.shape)

In [ ]:
# Extract color histograms from the blobs
def color_histogram_extraction(blobs_gray):
    num_bins = 32
    norm = cv2.NORM_L1
    
    color_hists = []
    for blob in blobs_gray:
        # Convert the blob to RGB format
        blob = cv2.cvtColor(blob, cv2.COLOR_GRAY2RGB)
        
        # Calculate the histograms
        red_hist = cv2.calcHist([blob], [0], None, [num_bins], [0, 256]).flatten()
        green_hist = cv2.calcHist([blob], [1], None, [num_bins], [0, 256]).flatten()
        blue_hist = cv2.calcHist([blob], [2], None, [num_bins], [0, 256]).flatten()

        # Normalize the histograms
        red_hist = cv2.normalize(red_hist, red_hist, norm_type=norm)
        green_hist = cv2.normalize(green_hist, green_hist, norm_type=norm)
        blue_hist = cv2.normalize(blue_hist, blue_hist, norm_type=norm)

        # Concatenate the histograms
        color_hist = np.concatenate((red_hist, green_hist, blue_hist))
        color_hists.append(color_hist)
    return np.array(color_hists)

# Extract color histograms from the grayscale image
print("Extracting color histograms...")
color_histograms = color_histogram_extraction(blobs_gray)
print("Extracted color histograms shape:", color_histograms.shape)

In [ ]:
# Get mean, std, and skewness of the blobs
def get_mean_std_skewness(images):
    color_feats = []
    
    for img in images:
        mean = np.mean(img)
        std = np.std(img)
        
        if std < 1e-5:  # Avoid division by zero
            skewness = 0
        else:
            skewness = skew(img.flatten())
        
        # Combine mean, std, and skewness into a single feature vector for the image
        color_feat = np.array([mean, std, skewness])
        color_feats.append(color_feat)
    
    return np.array(color_feats)

# Extract mean, std, and skewness from the blobs
print("Extracting mean, std, and skewness...")
mean_std_skew = get_mean_std_skewness(blobs_gray)
print("Extracted mean, std, and skewness shape:", mean_std_skew.shape)

In [ ]:
# Get LBP features from the blobs
def get_lbp_features(images, P=8, R=1):
    lbp_features = []
    for img in images:
        # Compute LBP
        lbp = local_binary_pattern(img, P, R, method='uniform')
        
        # Calculate the histogram of LBP
        hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, P + 3), density=True)
        
        # Append the histogram to the list
        lbp_features.append(hist)
    
    return np.array(lbp_features)

# Extract LBP features from the blobs
print("Extracting LBP features...")
lbp_features = get_lbp_features(blobs_gray)
print("Extracted LBP features shape:", lbp_features.shape)

In [ ]:
# Multithreaded Get Shape Features from the blobs
def get_shape_features_parallel(images):
    def process_image_shape(img):
        try:
            # Label the image
            labeled_img = label(img)

            # Get region properties
            props = regionprops(labeled_img)

            # Extract features for each region
            for prop in props:
                area = prop.area
                perimeter = prop.perimeter
                eccentricity = prop.eccentricity
                solidity = prop.solidity
                extent = prop.extent

            # Return the features
            return [area, perimeter, eccentricity, solidity, extent]
        except Exception as e:
            print(f"Error processing image: {e}")
            return [0, 0, 0, 0, 0]

    # Use joblib to parallelize the processing of images
    shape_features = Parallel(n_jobs=-1)(delayed(process_image_shape)(img) for img in images)
    return np.array(shape_features)

# Extract shape features from the blobs
print("Extracting shape features...")
shape_features = get_shape_features_parallel(blobs_gray)
print("Extracted shape features shape:", shape_features.shape)

In [ ]:
# Parallelize feature extraction for blobs
def extract_features_parallel(blobs):
    def process_blob(blob):
        # Extract features for a single blob
        hog_features = get_hog_features([blob])
        color_histograms = color_histogram_extraction([blob])
        mean_std_skew = get_mean_std_skewness([blob])
        lbp_features = get_lbp_features([blob])
        # shape_features = get_shape_features_parallel([blob])
        # Concatenate all features
        return np.concatenate([hog_features[0], color_histograms[0], mean_std_skew[0], lbp_features[0]])

    # Use joblib to parallelize the feature extraction
    features = Parallel(n_jobs=-1)(delayed(process_blob)(blob) for blob in blobs)
    return np.array(features)

# Example usage
features = extract_features_parallel(blobs_gray)
print(f"Extracted features shape: {features.shape}")

#### 3.1.5. Using these features, build a detection model to segregate tumor pathologies from non-tumor pathologies.

In [ ]:
# This model is just a placeholder to show how the helper functions from this part work
# The real model is under 3.2
# Based on the GridSearchCV testing we did (presented below), the best parameters for the SVC model are:
# C = 5, 'degree': 3, 'gamma': 0.001, kernel = 'rbf', class_weight = 'balanced'

param_grid = {'C': 5, 'gamma': 0.001}
rbf_clf = SVC(kernel='rbf', C=param_grid['C'], gamma=param_grid['gamma'])

models = [rbf_clf]
model_names = ['SVC (RBF)']

#### 3.1.6. The target label for tumors and non-tumors can be obtained from the corresponding csv file of the image.

In [ ]:
# Get the ground truth labels from the CSV file
def get_labels_from_csv(csv_data, skipped_blobs):
    # Extract the labels from the CSV file except for the skipped blobs
    labels = []
    for i, row in csv_data.iterrows():
        if i not in skipped_blobs and row['raw_classification'] == 'tumor':
            labels.append(1)
        elif i not in skipped_blobs:
            labels.append(0)
    return np.array(labels)

csv_data = pd.read_csv(path + 'csv/' + training_image_path + '.csv')
training_labels = get_labels_from_csv(csv_data, skipped_blobs)
print(f"Training labels: {training_labels}")
print(f"Training labels length: {len(training_labels)}")

In [ ]:
predictions = []
for model in models:
    # Fit the model on the training data
    model.fit(features, training_labels)

    # Predict on the training data
    pred = model.predict(features)
    predictions.append(pred)

    # Calculate accuracy
    accuracy = accuracy_score(training_labels, pred)
    print(f"Accuracy for {model_names[0]}: {accuracy:.4f}")

In [ ]:
# Use the labels and blob coordinates to draw the bounding boxes on the image
# If a tumor: draw a red bounding box
# If not a tumor: draw a blue bounding box

def draw_polylines_on_image(image, blob_coords, labels):
    draw_image = image.copy()
    # print(f"Number of blobs: {len(blob_coords)}")
    # print(f"Number of labels: {len(labels)}")
    for i, (xmin, ymin, xmax, ymax, coords_x, coords_y) in enumerate(blob_coords):
        
        points = np.column_stack((coords_x, coords_y))  # Create (x,y) coordinate pairs
        # Draw the bounding box
        if labels[i] == 1:
            color = (255, 0, 0)
        else:
            color = (0, 0, 255)
        cv2.polylines(draw_image, [points], isClosed=True, color=color, thickness=2)

    return draw_image

for model_name, pred in zip(model_names, predictions):
    # Draw bounding boxes on the image
    prepared_image_rgb = cv2.cvtColor(prepared_image, cv2.COLOR_GRAY2RGB)
    draw_image = draw_polylines_on_image(prepared_image_rgb, blob_coords, training_labels)
    ground_image = draw_polylines_on_image(prepared_image_rgb, blob_coords, pred)

    plt.figure(figsize=(7, 7))
    plt.subplot(1, 2, 1)
    plt.imshow(draw_image)
    plt.title(f'{model_name}\n Tumor (Red) vs Non-Tumor (Blue)')
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.imshow(ground_image)
    plt.title('Ground Truth Image\n Tumor (Red) vs Non-Tumor (Blue)')
    plt.axis('off')
    plt.show()

In [ ]:
# Given an image name, get the features and labels for that image
def get_features_and_label_for_image_name(image_name):
    # Load the image and mask
    img_gray, mask_gray = convert_img_and_mask_to_grayscale(image_name)
    
    # Get interest regions
    img_interest_regions = get_interest_regions(img_gray, mask_gray)
    
    # Remove background noise
    prepared_image = remove_background_noise(img_interest_regions)
    
    # Extract blobs and their coordinates
    csv_data = pd.read_csv(path + 'csv/' + image_name + '.csv')
    blobs_gray, _, skipped_blobs = extract_blobs(prepared_image, csv_data)
    
    # Extract features from the blobs
    features = extract_features_parallel(blobs_gray)
    
    # Get labels from CSV data
    training_labels = get_labels_from_csv(csv_data, skipped_blobs)
    return features, training_labels

features, labels = get_features_and_label_for_image_name(training_image_path)
print(f"Features shape: {features.shape}")
print(f"Number of labels : {len(labels)}")

Quick and dirty testing on a test image

In [ ]:
image_name1 = 'TCGA-A1-A0SP-DX1_id-5ea4095addda5f8398977ebc_left-11391_top-53955_bottom-54237_right-11688'
image1 = cv2.imread(path + 'rgb/' + image_name1 + '.png', cv2.IMREAD_COLOR)
image1 = cv2.cvtColor(image1, cv2.COLOR_BGR2RGB)
csv_data = pd.read_csv(path + 'csv/' + image_name1 + '.csv')
blobs1, blob_coords1, skip_blobs1 = extract_blobs(image1, csv_data)
features1, labels1 = get_features_and_label_for_image_name(image_name1)

predictions = []
for model in models:
    print(f"Predicting with model: {model}")
    pred = model.predict(features1)
    predictions.append(pred)
    print(f"Predictions shape: {pred.shape}")
    accuracy = accuracy_score(pred, labels1)
    print(f"Accuracy for model {model}: {accuracy:.4f}")

#### 3.1.7. The count for this image would be the number of tumor in the given image i.e, count the number of instances of tumor for the given image. The ground truth count can be obtained by reporting the number of instances of tumor from the corresponding csv file.

In [ ]:
# Count the number of instances of tumor for the given image
def count_tumor_instances_in_image(image_name, predictions, skipped_blobs):
    
    csv_data = pd.read_csv(path + 'csv/' + image_name + '.csv')
    ground_truth_labels = get_labels_from_csv(csv_data, skipped_blobs)
    
    ground_truth_tumor_count = np.sum(ground_truth_labels == 1)
    predicted_tumor_count = np.sum(predictions == 1)
    
    return ground_truth_tumor_count, predicted_tumor_count

tumor_count = np.sum(get_labels_from_csv(csv_data, skipped_blobs) == 1)
print(f"Number of tumor instances in the image (Ground Truth): {tumor_count}")

# Get the count of tumor instances for each model
tumor_counts = []
for i, model in enumerate(models):
    _, tumor_count = count_tumor_instances_in_image(image_name1, predictions[i], skipped_blobs)
    tumor_counts.append(tumor_count)
    print(f"Number of tumor instances in the image (Model {model}): {tumor_count}")
    ground_image = draw_polylines_on_image(image1, blob_coords1, get_labels_from_csv(csv_data, skipped_blobs))
    predicted_image = draw_polylines_on_image(image1, blob_coords1, predictions[i])
    
    plt.figure(figsize=(7, 7))
    plt.subplot(1, 2, 1)
    plt.imshow(ground_image)
    plt.title('Ground Truth')
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.imshow(predicted_image)
    plt.title('Predicted')
    plt.axis('off')
    plt.show()

#### 3.1.8. Repeat the steps mentioned above, to loop over all the images for the subjects in train subjects.csv to construct a python data frame with Xtrain as features and Ytrain as the target label of tumors and non-tumors. Similarly, create Xval and Yval for the images of subjects in val subjects.csv.

In [ ]:
# Given a subject ID, get all corresponding image names
def get_all_image_names_for_subject(subject_id):
    subject_imgs = []
    
    # Get all image names for the subject ID
    for img_name in os.listdir(path + 'rgb/'):
        if subject_id in img_name:
            subject_imgs.append(img_name.split('.')[0])
    return subject_imgs

subject_id = 'TCGA-A1-A0SP-DX1_id'
image_names = get_all_image_names_for_subject(subject_id)
print(f"Number of images for subject \'{subject_id}\': {len(image_names)}")

In [ ]:
# Given a subject ID, get all features and labels for all images
def get_features_and_labels_for_subject(subject_id):
    image_names = get_all_image_names_for_subject(subject_id)
    
    subject_features = []
    subject_labels = []
    for img_name in image_names:
        features, labels = get_features_and_label_for_image_name(img_name)
        subject_features.append(features)
        subject_labels.append(labels)
    return np.concatenate(subject_features), np.concatenate(subject_labels)

# Get features and labels for the subject
subject_id = 'TCGA-A1-A0SP-DX1_id'
features, labels = get_features_and_labels_for_subject(subject_id)
print(f"Features shape: {features.shape}")
print(f"Labels shape: {labels.shape}")

Get the features and labels of every blob in the datasets

In [ ]:
scaler = StandardScaler()

# Function to process a single subject
def process_subject(subject_id):
    features, labels = get_features_and_labels_for_subject(subject_id)
    return features, labels

# For all subjects in a csv file, create a python data frame with the features and labels
def get_features_and_labels_for_all_subjects_in_csv_parallel(subjects_file):
    subjects = pd.read_csv(subjects_file)
    subject_ids = subjects['subject_id'].tolist()
    
    # Use joblib to parallelize the processing of subjects
    results = Parallel(n_jobs=-1)(
        delayed(process_subject)(subject_id) for subject_id in subject_ids
    )
    
    # Combine the results
    all_subject_features = np.concatenate([res[0] for res in results])
    all_subject_labels = np.concatenate([res[1] for res in results])
    
    return all_subject_features, all_subject_labels

print("Getting features and labels for all subjects in the training dataset...")
Xtrain, ytrain = get_features_and_labels_for_all_subjects_in_csv_parallel(path + 'splits/train_subjects.csv')
Xtrain = scaler.fit_transform(Xtrain)  # Standardize the features
print(f"Xtrain shape: {Xtrain.shape}")
print(f"ytrain shape: {ytrain.shape}")

print("Getting features and labels for all subjects in the validation dataset...")
Xval, yval = get_features_and_labels_for_all_subjects_in_csv_parallel(path + 'splits/validate_subjects.csv')
Xval = scaler.transform(Xval)  # Standardize the features using the same scaler as for training
print(f"Xval shape: {Xval.shape}")
print(f"yval shape: {yval.shape}")

Apply PCA to the features to remove noise and reduce the feature space

In [ ]:
# Apply PCA to the features

scaler_pca = StandardScaler()

def apply_pca(X, n_components=0.95):
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X)
    return X_pca, pca

# Apply PCA to the training and validation features
print("Applying PCA to training features...")
Xtrain_pca, pca = apply_pca(Xtrain, n_components=0.95)
print(f"Xtrain_pca shape: {Xtrain_pca.shape}")

print("Applying PCA to validation features...")
Xval_pca = pca.transform(Xval)  # Use the same PCA model as for training
print(f"Xval_pca shape: {Xval_pca.shape}")

# Reapply scaler to the PCA features
Xtrain_pca = scaler_pca.fit_transform(Xtrain_pca)
Xval_pca = scaler_pca.transform(Xval_pca)

Save functions to store the features and labels to avoid having to get everything again

In [ ]:
# Save the features and labels to a CSV file
def save_features_and_labels_to_csv(X, y, filename):
    # Check if the file already exists and remove it
    if os.path.exists(filename):
        print(f"File {filename} already exists. Removing it.")
        os.remove(filename)
        
    # Create a DataFrame with features and labels
    df = pd.DataFrame(X)
    df['label'] = y
    df.to_csv(filename, index=False)
    print(f"Saved features and labels to {filename}")
    print(f"Shape of features: {X.shape}")

# Save the training and validation features and labels to CSV files
save_features_and_labels_to_csv(Xtrain_pca, ytrain, path + 'datasets/train_dataset.csv')
save_features_and_labels_to_csv(Xval_pca, yval, path + 'datasets/validate_dataset.csv')

def save_scaler_to_file(scaler, filename):
    # Check if the file already exists and remove it
    if os.path.exists(filename):
        print(f"File {filename} already exists. Removing it.")
        os.remove(filename)
        
    # Save the scaler to a file
    dump(scaler, open(filename, 'wb'))
    print(f"Saved scaler to {filename}")

# Save the scalers to files
save_scaler_to_file(scaler, path + 'datasets/scaler.pkl')
save_scaler_to_file(scaler_pca, path + 'datasets/scaler_pca.pkl')

# Save the PCA model to a file
print("Saving PCA model to file...")
if os.path.exists(path + 'datasets/pca_model.pkl'):
    print(f"File {path + 'datasets/pca_model.pkl'} already exists. Removing it.")
    os.remove(path + 'datasets/pca_model.pkl')
dump(pca, open(path + 'datasets/pca_model.pkl', 'wb'))

Load functions to get previous datasets

In [ ]:
# Load the dataset from a CSV file
def load_dataset_from_csv(filename):
    df = pd.read_csv(filename)
    X = df.iloc[:, :-1].values  # All columns except the last one
    y = df.iloc[:, -1].values   # Last column
    print(f"Loaded dataset shape: {X.shape}, labels shape: {y.shape}")
    return X, y

# Load the training and validation datasets
Xtrain, ytrain = load_dataset_from_csv(path + 'datasets/train_dataset.csv')
Xval, yval = load_dataset_from_csv(path + 'datasets/validate_dataset.csv')

# Load the scalers and PCA from files
print("Loading scalers from files...")
scaler = load(open(path + 'datasets/scaler.pkl', 'rb'))
scaler_pca = load(open(path + 'datasets/scaler_pca.pkl', 'rb'))

print("Standardizing training and validation datasets...")
Xtrain = scaler_pca.transform(Xtrain)  # Standardize the features
Xval = scaler_pca.transform(Xval)  # Standardize the features using the same scaler as for training

print("Loading PCA model from file...")
pca = load(open(path + 'datasets/pca_model.pkl', 'rb'))

### 3.2. Classification Framework

#### Helper functions

Check the class balance of the datasets 

In [ ]:
# Get the unique labels
dataset_labels = np.unique(np.concatenate((ytrain, yval)))
unique_labels = np.unique(dataset_labels)
print(f"Unique labels in the dataset: {unique_labels}")

# Per dataset
# Get the unique labels for training and validation sets
label_counts_train = {label: np.sum(ytrain == label) for label in unique_labels}
label_counts_val = {label: np.sum(yval == label) for label in unique_labels}
print(f"Label counts in the training set: {label_counts_train}")
print(f"Label counts in the validation set: {label_counts_val}")

# Get the number of instances for each label
label_counts = {label: np.sum(ytrain == label)+np.sum(yval==label) for label in unique_labels}
print(f"Label counts in the dataset: {label_counts}")

# Sort the labels by count
sorted_labels = sorted(label_counts.items(), key=lambda x: x[1])
print(f"Sorted labels by count: {sorted_labels}")

for label, count in sorted_labels:
    print(f"Label: {label}, Count: {count}")

# Check for an imbalance in the dataset
class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(ytrain), y=ytrain)
print(f"Class weights: {class_weights}")

Getting the optimal hyperparameters for the SVC model using GridSearchCV

In [ ]:
# Downsample the training data with stratification
Xtrain_small, _, ytrain_small, _ = train_test_split(
    Xtrain, ytrain, train_size=10000, stratify=ytrain, random_state=42
)

class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(ytrain_small), y=ytrain_small)
print(f"Class weights for the downsampled training set: {class_weights}")

Xtrain_small = scaler_pca.transform(Xtrain_small)  # Standardize the features

print(f"Xtrain_small shape: {Xtrain_small.shape}")
print(f"ytrain_small shape: {ytrain_small.shape}")

param_grid = {
    'C': [0.1, 1, 2, 5, 10],
    'gamma': [0.001, 0.01, 0.1, 1, 'auto', 'scale']
}

# Try for RBF SVM
print("Tuning hyperparameters for RBF SVM...")
grid_rbf = GridSearchCV(SVC(kernel='rbf', class_weight='balanced'), param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_rbf.fit(Xtrain_small, ytrain_small)
print("Best parameters found (for RBF): ", grid_rbf.best_params_)

Train the model on the full training dataset (Long)

In [ ]:
class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(ytrain), y=ytrain)
print(f"Class weights for the full training set: {class_weights}")

# Using the best params, train the SVM models on the entire training dataset
best_params_rbf = grid_rbf.best_params_
rbf_clf = SVC(kernel='rbf', C=best_params_rbf['C'], gamma=best_params_rbf['gamma'], class_weight='balanced')

print(f"Training features shape: {Xtrain.shape}")
print(f"Training labels shape: {ytrain.shape}")

# Train the model on the entire training dataset
print("Training SVM models with the best parameters...")
print(f"RBF SVM")
rbf_clf.fit(Xtrain, ytrain)

models = [rbf_clf]
model_names = ['RBF SVM (Full New)']

Train the model on a smaller batch of the training dataset (Short)

In [ ]:
Xtrain_small, _, ytrain_small, _ = train_test_split(
    Xtrain, ytrain, train_size=0.25, stratify=ytrain, random_state=42
)

class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(ytrain_small), y=ytrain_small)
print(f"Class weights for the downsampled training set: {class_weights}")

Xtrain_small = scaler_pca.transform(Xtrain_small)  # Standardize the features using the same scaler as for training
print(f"Xtrain_small shape: {Xtrain_small.shape}")

print("Training SVM models with the best parameters...")
# Using the best params, train the SVM models on a smaller dataset
best_params_rbf = grid_rbf.best_params_
rbf_clf = SVC(kernel='rbf', C=best_params_rbf['C'], gamma=best_params_rbf['gamma'], class_weight='balanced')
rbf_clf.fit(Xtrain_small, ytrain_small)
print("Training complete.")

models = [rbf_clf]
model_names = ['RBF SVM (PCA Quarter)']

Save functions to save models

In [ ]:
# Save the models to a pickle file
# If the models folder doesn't exist, create it
if not os.path.exists(path + 'models'):
    os.makedirs(path + 'models')

def save_models_to_pickle(models, model_names, folder):
    for model, name in zip(models, model_names):
        filename = path + folder + name.replace(' ', '_') + '.pkl'
        # Check if the file already exists and remove it
        if os.path.exists(filename):
            print(f"File {filename} already exists. Removing it.")
            os.remove(filename)
            
        # Save the model to a pickle file
        with open(filename, 'wb') as f:
            pickle.dump(model, f)
        print(f"Model saved to {filename}")
save_models_to_pickle(models, model_names, 'models/')

Load functions to retrieve previous models

In [ ]:
# Load the models from a pickle file
def load_models_from_pickle(filename):
    with open(filename, 'rb') as f:
        model = load(f)
    return model

models = []
model_names = []
pca = load(open(path + 'datasets/pca_model.pkl', 'rb'))

# Load the models from the pickle files in the models folder
for filename in os.listdir(path + 'models/'):
    if filename.endswith('.pkl'):
        model = load_models_from_pickle(path + 'models/' + filename)
        models.append(model)
        model_names.append(filename.replace('.pkl', '').replace('_', ' '))
        print(f"Model loaded from {filename}")
        print(f"Model name: {model_names[-1]}")

A helper function to run an entire detection on the image using a trained model

In [ ]:
# Run an entire detection on the image using a trained model
def do_full_detection(model, pca, image_name):
    
    # Load the original image
    img = cv2.imread(path + 'rgb/' + image_name + '.png')
    csv_data = pd.read_csv(path + 'csv/' + image_name + '.csv')
    
    # Load the image and mask
    img_gray, mask_gray = convert_img_and_mask_to_grayscale(image_name)
    
    # Get the interest regions
    img_interest_regions = get_interest_regions(img_gray, mask_gray)
    
    # Remove background noise
    prepared_image = remove_background_noise(img_interest_regions)
    
    # Extract blobs from the image
    blobs_gray_full, blob_coords_full, skipped_blobs = extract_blobs(prepared_image, csv_data)
    
    # Extract features from the blobs
    features_full = extract_features_parallel(blobs_gray_full)
    features_full = scaler.transform(features_full)  # Standardize the features using the same scaler as for training
    features_full = pca.transform(features_full)  # Apply PCA to the features
    features_full = scaler_pca.transform(features_full)  # Standardize the features using the same scaler as for training
    
    # Predict the class of each blob using the trained model
    predictions = model.predict(features_full)

    ground_truth_num_tumors, predicted_num_tumors = count_tumor_instances_in_image(image_name, predictions, skipped_blobs)

    ground_truth_labels = get_labels_from_csv(csv_data, skipped_blobs)    
    accuracy = accuracy_score(ground_truth_labels, predictions)
    
    # Draw bounding boxes on the image
    ground_image = draw_polylines_on_image(img, blob_coords_full, ground_truth_labels)
    predicted_image = draw_polylines_on_image(img, blob_coords_full, predictions)
    
    return ground_truth_num_tumors, predicted_num_tumors, predictions, blobs_gray_full, blob_coords_full, accuracy, ground_image, predicted_image

# Test the function with the testing image
testing_image_path = 'TCGA-A2-A0D2-DX1_id-5ea40b0eddda5f8398997d45_left-100214_top-23322_bottom-23602_right-100471' # This image has a mix of tumors and non-tumors

# Run the full detection on the testing image using each model
for model, model_name in zip(models, model_names):
    print("--------------------------------------------")
    print(f"Running full detection with model: {model_name}")
    num_tumors, predicted_num_tumors, predictions, blobs_gray_full, blob_coords_full, accuracy, ground_image, predicted_image = do_full_detection(model, pca, testing_image_path)
    print(f"Predictions: {predictions}")
    print(f"Number of tumors (Ground Truth): {num_tumors}")
    print(f"Number of tumors (Predicted): {predicted_num_tumors}")
    print(f"Accuracy: {accuracy:.4f}")
    
    plt.figure(figsize=(7, 7))
    plt.subplot(1, 2, 1)
    plt.imshow(ground_image)
    plt.title('Ground Truth Labels')
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.imshow(predicted_image)
    plt.title('Predicted Labels')
    plt.axis('off')
    plt.show()

Quick and dirty testing

In [ ]:
# Predict on the validation set
predictions_val = []
for model, model_name in zip(models, model_names):
    print(f"Predicting on validation set with model: {model}")
    print(f"Xval shape: {Xval.shape}")
    pred = model.predict(Xval)
    predictions_val.append(pred)
    print(f"Predictions shape: {pred.shape}")
    
    accuracy = accuracy_score(yval, pred)
    print(f"Accuracy for model {model_name}: {accuracy:.4f}")

In [ ]:
# Count the number of tumors that were detected in the validation set
print("Counting the number of tumors in the validation set...")
ground_truth_non_tumor_count = np.sum(yval == 0)
ground_truth_tumor_count = np.sum(yval == 1)
print(f"Number of non-tumors in the validation set (Ground Truth): {ground_truth_non_tumor_count}")
print(f"Number of tumors in the validation set (Ground Truth): {ground_truth_tumor_count}")
for i, (model, model_name) in enumerate(zip(models, model_names)):
    non_tumor_count = np.sum(predictions_val[i] == 0)
    print(f"Number of non-tumors in the validation set (Predicted) with model {model_name}: {non_tumor_count}")
    tumor_count = np.sum(predictions_val[i] == 1)
    print(f"Number of tumors in the validation set (Predicted) with model {model_name}: {tumor_count}")
    
    mean_tumor_count = tumor_count / ground_truth_tumor_count * 100
    print(f"Mean tumor count in the validation set (Predicted) with model {model_name}: {mean_tumor_count:.4f}")

Feature Analysis

In [ ]:
# Function to evaluate a single feature set
def evaluate_feature(name, features, labels):
    print(f"Evaluating feature set: {name}")
    accuracy = evaluate_feature_set(features, labels)
    print(f"Accuracy for {name}: {accuracy:.4f}")
    return name, accuracy

def extract_all_features(images):
    return {
        'HOG': get_hog_features(images),
        'ColorHist': color_histogram_extraction(images),    
        'MeanStdSkew': get_mean_std_skewness(images),
        'LBP': get_lbp_features(images),
        # 'Shape': get_shape_features_parallel(images) # This feature is commented out due to performance issues (it takes a long time to run)
    }

def evaluate_feature_set(X, y):
    # Downsample the data to 80% of the original size
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    param_grid = {'C': 5, 'degree': 2, 'gamma': 0.001}
    clf = SVC(kernel='rbf', C=param_grid['C'], gamma=param_grid['gamma'], class_weight='balanced')
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_val_scaled)

    return accuracy_score(y_val, y_pred)

path = './QC/'

subjects = pd.read_csv(path + 'splits/validate_subjects.csv')
subject_ids = subjects['subject_id'].tolist()

blobs = []
labels = []

for subject_id in subject_ids:
    image_names = get_all_image_names_for_subject(subject_id)

    for image_name in image_names:
        prepared_image = cv2.imread(path + 'rgb/' + image_name + '.png', cv2.IMREAD_GRAYSCALE)
        csv_data = pd.read_csv(path + 'csv/' + image_name + '.csv')

        blobs_gray, _, skipped_blobs = extract_blobs(prepared_image, csv_data)

        blobs.extend(blobs_gray)
        labels.extend(get_labels_from_csv(csv_data, skipped_blobs))

hs = []
ws = []
for blob in blobs:
    hs.append(blob.shape[0])
    ws.append(blob.shape[1])
print(f"Max height: {max(hs)}, Max width: {max(ws)}")
print(f"Min height: {min(hs)}, Min width: {min(ws)}")
print(f"Mean height: {np.mean(hs)}, Mean width: {np.mean(ws)}")

In [ ]:
print(f"Extracting features from {len(blobs)} images...")
features_dict = extract_all_features(blobs)
accuracies = {}

# Run the evaluation in parallel
results = Parallel(n_jobs=-1)(delayed(evaluate_feature)(name, features, labels) for name, features in features_dict.items())

# Collect the results into the accuracies dictionary
accuracies = dict(results)

# Print the results
for name, accuracy in accuracies.items():
    print(f"Feature set: {name}, Accuracy: {accuracy:.4f}")

plt.bar(accuracies.keys(), accuracies.values(), color='skyblue')
plt.ylabel('Accuracy')
plt.title('Feature Set Evaluation')
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.show()

#### 3.2.1. Report performance metrics of the detection model for both training and validation images.


In [ ]:
# Function to get the classification report and confusion matrix
def report_performance_metrics(model, X_train, y_train, X_val, y_val):

    # Predictions on training data
    y_train_pred = model.predict(X_train)
    
    # Predictions on validation data
    y_val_pred = model.predict(X_val)
    
    # Training metrics
    print("Performance Metrics for Training Data:")
    print("Accuracy:", accuracy_score(y_train, y_train_pred))
    print("Classification Report:")
    print(classification_report(y_train, y_train_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_train, y_train_pred))
    
    # Validation metrics
    print("Performance Metrics for Validation Data:")
    print("Accuracy:", accuracy_score(y_val, y_val_pred))
    print("Classification Report:")
    print(classification_report(y_val, y_val_pred))
    print("Confusion Matrix:")
    confusion_M = confusion_matrix(y_val, y_val_pred)
    print(confusion_M)

    # Display the confusion matrix
    plt.figure(figsize=(5, 5))
    sns.heatmap(confusion_M, annot=True, fmt='d', cmap='Blues', cbar=False, 
                xticklabels=['Non-tumor', 'Tumor'], yticklabels=['Non-tumor', 'Tumor'])
    plt.xticks()
    plt.title('Linear SVM Confusion Matrix')
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.show()

# Report performance metrics for each model
for i, model in enumerate(models):
    print(f"Performance Metrics for Model {model_names[i]}:")
    report_performance_metrics(model, Xtrain, ytrain, Xval, yval)
    print("\n" + "="*50 + "\n")
    break # Remove this line to run for all models

#### 3.2.2. After detecting the tumor for all validation images, count the number of detector tumors in the image. Report the root mean squared error (RMSE) for this task of counting tumors.

In [ ]:
# Function to calculate the mean absolute error between ground truth and predicted tumor counts
def calculate_rmse(model, subjects_csv_name):
    predicted_counts = []
    ground_truth_counts = []
    differences = []
    subjects = []

    # Iterate through the validation subject csv
    for i, row in pd.read_csv(subjects_csv_name).iterrows():
        subjects.append(row['subject_id'])
    
    validation_images = []
    for subject in subjects:
        # Get all image names for the subject
        images = get_all_image_names_for_subject(subject)
        validation_images.extend(images)

    for image_name in validation_images:
        
        # Get the ground truth and predicted tumor counts for the image
        ground_truth_tumor_count, predicted_tumor_count, _, _, _, _, _, _ = do_full_detection(model, pca, image_name)
        
        ground_truth_counts.append(ground_truth_tumor_count)
        predicted_counts.append(predicted_tumor_count)
        differences.append(predicted_tumor_count - ground_truth_tumor_count)
    
    # Calculate RMSE between ground truth and predicted counts
    rmse = np.sqrt(mean_squared_error(ground_truth_counts, predicted_counts))
    
    # Print the final RMSE culmination
    print("Final RMSE culmination:", rmse)
    
    return rmse, differences, predicted_counts, ground_truth_counts

# Display the RMSE for each model
for i, model in enumerate(models):
    print(f"RMSE for Model {model_names[i]}:")
    rmse, difference, _, _ = calculate_rmse(model, path+'splits/validate_subjects.csv')
    print(f"RMSE: {rmse:.4f}")
    print("\n" + "="*50 + "\n")
    
    # Plot the differences as a bar chart x axis should be the difference and y axis should be the number of instances
    plt.figure(figsize=(10, 5))
    plt.hist(difference, bins=20, color='blue', alpha=0.7)
    plt.title(f'Differences in Tumor Counts for Model {model_names[i]}')
    plt.xlabel('Difference in Tumor Counts')
    plt.ylabel('Number of Instances')
    plt.grid(axis='y', alpha=0.75)
    plt.axvline(x=0, color='red', linestyle='--', label='Ground Truth')
    plt.legend()
    plt.tight_layout()
    plt.show()
    break # Remove this line to run for all models

#### 3.2.3. Is there a difference in the performance if you use the ‘original image’ in Fig. 2 (i.e. the unmasked image (Step 1)) instead of only classifying each of the provided pathological regions (i.e. the masked regions)? Provide a qualitative and quantitative discussion of your results. (Hint: To answer this you will need to recreate Xtrain, Ytrain, Xval, and Yval. You are required to use the features from the original image for this task.)

In [ ]:
# Given the original image, break it down into grids of n x n pixels
def break_image_into_grids(image, grid_size=20):
    
    height, width = image.shape
    # print(f"Image shape: {image.shape}")
    
    # Print the number of rows and columns in the image
    # num_rows = height // grid_size
    # num_cols = width // grid_size
    # print(f"Number of rows: {num_rows}, Number of columns: {num_cols}")
    
    grids = []
    grid_coords = []
    step = grid_size // 2
    for i in range(0, height, step):
        for j in range(0, width, step):
            if j + grid_size > width or i + grid_size > height:
                # print(f"Skipping grid at ({i}, {j}) as it exceeds image dimensions")
                continue
            grid = image[i:i+grid_size, j:j+grid_size]
        
            # Append the grid to the list
            grids.append(grid)
            # Append the coordinates of the grid to the list
            min_x = j
            min_y = i
            max_x = j + grid_size
            max_y = i + grid_size
            grid_coords.append((min_x, min_y, max_x, max_y))
            # print(f"Grid coordinates: minx:{min_x}, miny:{min_y}, maxx:{max_x}, maxy:{max_y}")
        
    # print(f"Number of grids: {len(grids)}")
    
    return grids, grid_coords

def get_pixel_labels(image_name):
    # Load the CSV file for the image
    df = pd.read_csv(path + 'csv/' + image_name + '.csv')
    
    # Load the image
    img = cv2.imread(path + 'rgb/' + image_name + '.png')
    
    # Initialize a label grid with zeros
    pixel_labels = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
    
    # Assign labels based on the bounding box annotations
    for _, row in df.iterrows():
        xmin, ymin, xmax, ymax = row["xmin"], row["ymin"], row["xmax"], row["ymax"]
        if row["raw_classification"] == "tumor":
            # Assuming the class_id for tumor is 1
            pixel_labels[ymin:ymax, xmin:xmax] = 1
        else:
            pixel_labels[ymin:ymax, xmin:xmax] = 2
    
    return pixel_labels

# Verify if the majority of the pixels in the grid are tumor pixels
def get_grid_label(grid, coords, pixel_labels):
    
    non_tumor_pixels = 0
    tumor_pixels = 0
    skip_pixels = 0
    # Count the number of tumor pixels in the grid
    xmin, ymin, xmax, ymax = coords
    for i in range(ymin, ymax):
        for j in range(xmin, xmax):
            if pixel_labels[i, j] == 1:
                tumor_pixels += 1
            elif pixel_labels[i, j] == 2:
                non_tumor_pixels += 1
            else:
                skip_pixels += 1

    # Assuming that the tumor pixels are labeled as 1
    total_pixels = grid.size
    # print(f"Grid coordinates: {coords}")
    # print(f"Non-tumor pixels in grid: {non_tumor_pixels}")
    # print(f"Tumor pixels in grid: {tumor_pixels}")
    # print(f"Total pixels in grid: {total_pixels}")
    
    # Check if the majority of the pixels are tumor or non-tumor pixels
    if tumor_pixels == 0 and non_tumor_pixels == 0:
        return 0 # Unknown
    elif tumor_pixels >= non_tumor_pixels and tumor_pixels > total_pixels // 2:
        return 1 # Tumor
    elif non_tumor_pixels > tumor_pixels and non_tumor_pixels > total_pixels // 2:
        return 2 # Non-tumor
    else:
        return 0 # Unknown

In [ ]:
# Function to process a single grid
def process_grid(grid, grid_local_coords, pixel_labels):
    grid_label = get_grid_label(grid, grid_local_coords, pixel_labels)
    # if grid_label == 0:
        # return None, None, None  # Skip the grid if it is unknown
    grid_features = extract_features_parallel([grid])[0]  # Extract features from the grid
    return grid_features, grid_label, grid_local_coords

# Parallelize grid processing for an image
def get_grid_features_and_labels_for_image_parallel(image_name, grid_size=20):
    input_image = cv2.imread(path + 'rgb/' + image_name + '.png', cv2.IMREAD_GRAYSCALE)
    grids, grid_coords = break_image_into_grids(input_image, grid_size)
    pixel_labels = get_pixel_labels(image_name)

    # Use joblib to parallelize grid processing
    results = Parallel(n_jobs=-1)(
        delayed(process_grid)(grid, coords, pixel_labels) for grid, coords in zip(grids, grid_coords)
    )

    # Separate features and labels
    grid_features = np.array([res[0] for res in results if res[0] is not None])
    grid_labels = np.array([res[1] for res in results if res[1] is not None])
    coords_to_keep = [res[2] for res in results if res[2] is not None]

    return grid_features, grid_labels, coords_to_keep

# Example usage
train_grid_features, train_grid_labels, train_grid_coords = get_grid_features_and_labels_for_image_parallel(testing_image_path, grid_size=60)
print(f"Grid features shape: {train_grid_features.shape}, Grid labels shape: {len(train_grid_labels)}")

In [ ]:
# Display the grid as tumors and non-tumors
def display_grid_labels(image_name, grid_size=20):
    # Count the number of tumor
    tumor_count = 0

    # Load the original image
    img = cv2.imread(path + 'rgb/' + image_name + '.png', cv2.IMREAD_GRAYSCALE)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    # Break the image into grids
    grids, grid_coords = break_image_into_grids(img, grid_size)
    print(f"Number of grids: {len(grids)}")
    
    # Get the pixel labels for the image
    pixel_labels = get_pixel_labels(image_name)
    
    # Draw the grid labels on the image
    for grid, coords in zip(grids, grid_coords):
        grid_label = get_grid_label(grid, coords, pixel_labels)
        xmin, ymin, xmax, ymax = coords
        if grid_label == 1:
            color = (255, 0, 0) # Red for tumor
            tumor_count += 1
        elif grid_label == 2:
            color = (0, 0, 255) # Blue for non-tumor
        else:
            color = (0, 255, 0) # Green for unknown
        cv2.rectangle(img_rgb, (xmin, ymin), (xmax, ymax), color, 1)

    plt.imshow(img_rgb)
    plt.title('Grid Labels - Tumor (Red), Non-tumor (Blue), Unknown (Green)')
    plt.axis('off')
    plt.show()

    print(f"Number of tumors in grid: {tumor_count}")

    return img_rgb

# Display the grid labels on the image
grid_image = display_grid_labels(testing_image_path, grid_size=60)

In [ ]:
# Display the grid labels on the image with the original image using the pixel labels
def display_grid_labels_with_pixel_labels(image_name):
    # Load the original image
    img = cv2.imread(path + 'rgb/' + image_name + '.png', cv2.IMREAD_GRAYSCALE)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    
    # Get the pixel labels for the image
    pixel_labels = get_pixel_labels(image_name)
    
    # Draw the pixel labels on the image
    for i in range(pixel_labels.shape[0]):
        for j in range(pixel_labels.shape[1]):
            if pixel_labels[i, j] == 1:
                color = (255, 0, 0) # Red for tumor
            elif pixel_labels[i, j] == 2:
                color = (0, 0, 255) # Blue for non-tumor
            else:
                color = (0, 255, 0)
            cv2.circle(img_rgb, (j, i), 1, color, -1)
    return img_rgb

# Display the grid labels on the image with the original image using the pixel labels
grid_image_pixel_labels = display_grid_labels_with_pixel_labels(testing_image_path)

# Display the ground truth pixel labels on the image next to the grid labels
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(grid_image)
plt.title('Grid Labels\nTumor (Red), Non-tumor (Blue), Unknown (Green)')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(grid_image_pixel_labels)
plt.title('Ground Truth Pixel Labels')
plt.axis('off')
plt.show()

In [ ]:
# Function to process a single subject
def process_grid_subject(subject_id):
    images_names = get_all_image_names_for_subject(subject_id)
    subject_features = []
    subject_labels = []
    subject_coords = []
    
    for image_name in images_names:
        features, labels, coords = get_grid_features_and_labels_for_image_parallel(image_name, grid_size=60) # Change grid size if needed -> 20 gives Xtrain shape: (436967, 1878) / Xval shape: (121016, 1878) (which is huge)
        subject_features.append(features)
        subject_labels.append(labels)
        subject_coords.append(coords)
    
    return np.concatenate(subject_features), np.concatenate(subject_labels), np.concatenate(subject_coords)

# Parallelize processing for all subjects in a CSV file
def get_grid_features_and_labels_for_all_subjects_in_csv_parallel(subjects_file):
    subjects = pd.read_csv(subjects_file)
    subject_ids = subjects['subject_id'].tolist()

    # Use joblib to parallelize the processing of subjects
    results = Parallel(n_jobs=-1)(
        delayed(process_grid_subject)(subject_id) for subject_id in subject_ids
    )

    # Combine the results
    all_subject_features = np.concatenate([res[0] for res in results])
    all_subject_labels = np.concatenate([res[1] for res in results])
    all_subject_coords = np.concatenate([res[2] for res in results])

    return all_subject_features, all_subject_labels, all_subject_coords

# Example usage
grid_scaler = StandardScaler()

print("Getting features and labels for all subjects in the training dataset...")
grid_Xtrain, grid_ytrain, train_coords = get_grid_features_and_labels_for_all_subjects_in_csv_parallel(path + 'splits/train_subjects.csv')
grid_Xtrain = grid_scaler.fit_transform(grid_Xtrain)  # Standardize the features
print(f"Xtrain shape: {grid_Xtrain.shape}, ytrain shape: {grid_ytrain.shape}")

print("Getting features and labels for all subjects in the validation dataset...")
grid_Xval, grid_yval, val_coords = get_grid_features_and_labels_for_all_subjects_in_csv_parallel(path + 'splits/validate_subjects.csv')
grid_Xval = grid_scaler.transform(grid_Xval)  # Standardize the features using the same scaler as for training
print(f"Xval shape: {grid_Xval.shape}, yval shape: {grid_yval.shape}")

In [ ]:
# Apply PCA to the features
grid_pca = PCA(n_components=0.95)
grid_pca_scaler = StandardScaler()

grid_Xtrain_pca = grid_pca.fit_transform(Xtrain)
print(f"Xtrain_pca shape: {grid_Xtrain_pca.shape}")

grid_Xval_pca = grid_pca.transform(Xval)  # Use the same PCA model as for training
print(f"Xval_pca shape: {grid_Xval_pca.shape}")

grid_Xtrain_pca = grid_pca_scaler.fit_transform(grid_Xtrain_pca)  # Standardize the features using the same scaler as for training
grid_Xval_pca = grid_pca_scaler.transform(grid_Xval_pca)  # Standardize the features using the same scaler as for training

In [ ]:
# Save the features, labels and coords in CSV files
def save_grid_features_and_labels_to_csv(X, y, coords, filename):
    # Check if the file already exists and remove it
    if os.path.exists(filename):
        print(f"File {filename} already exists. Removing it.")
        os.remove(filename)
        
    # Create a DataFrame with features, labels and coords
    df = pd.DataFrame(X)
    df['label'] = y
    df['coords'] = list(coords)
    df.to_csv(filename, index=False)
    print(f"Saved grid features and labels to {filename}")
    print(f"Shape of features: {X.shape}")

# Save the training features, labels and coords in CSV files
save_grid_features_and_labels_to_csv(grid_Xtrain_pca, grid_ytrain, train_coords, path + 'second_datasets/train_dataset.csv')

# Save the validation features, labels and coords in CSV files
save_grid_features_and_labels_to_csv(grid_Xval_pca, grid_yval, val_coords, path + 'second_datasets/validate_dataset.csv')

# Save the scalers to a file
print("Saving scalers to file...")
if os.path.exists(path + 'second_datasets/scaler.pkl'):
    print(f"File {path + 'second_datasets/scaler.pkl'} already exists. Removing it.")
    os.remove(path + 'second_datasets/scaler.pkl')
dump(scaler, open(path + 'second_datasets/scaler.pkl', 'wb'))
if os.path.exists(path + 'second_datasets/scaler_pca.pkl'):
    print(f"File {path + 'second_datasets/scaler_pca.pkl'} already exists. Removing it.")
    os.remove(path + 'second_datasets/scaler_pca.pkl')
dump(grid_pca_scaler, open(path + 'second_datasets/scaler_pca.pkl', 'wb'))

# Save the PCA model to a file
print("Saving PCA model to file...")
if os.path.exists(path + 'second_datasets/pca_model.pkl'):
    print(f"File {path + 'second_datasets/pca_model.pkl'} already exists. Removing it.")
    os.remove(path + 'second_datasets/pca_model.pkl')
dump(grid_pca, open(path + 'second_datasets/pca_model.pkl', 'wb'))

In [ ]:
# Load the dataset from a CSV file
def load_grid_dataset_from_csv(filename):
    df = pd.read_csv(filename)
    X = df.iloc[:, :-2].values  # All columns except the last two
    y = df.iloc[:, -2].values   # Second to last column
    coords = df.iloc[:, -1].values  # Last column
    print(f"Loaded dataset shape: {X.shape}, labels shape: {y.shape}, coords shape: {coords.shape}")
    return X, y, coords

# Load the training and validation datasets
grid_Xtrain, grid_ytrain, train_coords = load_grid_dataset_from_csv(path + 'second_datasets/train_dataset.csv')
print(f"Xtrain shape: {grid_Xtrain.shape}, ytrain shape: {grid_ytrain.shape}, train_coords shape: {train_coords.shape}")
grid_Xval, grid_yval, val_coords = load_grid_dataset_from_csv(path + 'second_datasets/validate_dataset.csv')
print(f"Xval shape: {grid_Xval.shape}, yval shape: {grid_yval.shape}, val_coords shape: {val_coords.shape}")

# Load the scalers from files
print("Loading scalers from files...")
grid_scaler = load(open(path + 'second_datasets/scaler.pkl', 'rb'))
grid_pca_scaler = load(open(path + 'second_datasets/scaler_pca.pkl', 'rb'))

# Load the PCA model from a file
print("Loading PCA model from file...")
grid_pca = load(open(path + 'second_datasets/pca_model.pkl', 'rb'))

In [ ]:
# Downsample the training data with stratification
grid_Xtrain_small, _, grid_ytrain_small, _ = train_test_split(
    grid_Xtrain, grid_ytrain, train_size=0.1, stratify=grid_ytrain, random_state=42
)

class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(grid_ytrain_small), y=grid_ytrain_small)
print(f"Class weights for the downsampled training set: {class_weights}")

grid_Xtrain_small = grid_pca_scaler.transform(grid_Xtrain_small)  # Standardize the features using the same scaler as for training
print(f"Xtrain_small shape: {grid_Xtrain_small.shape}")
print(f"ytrain_small shape: {grid_ytrain_small.shape}")

# Use the same parameters as before for the model
params = rbf_clf.get_params()
print(f"Parameters: {params}")

# Train the a new model on the downsampled dataset
print("Training SVM models with the best parameters...")
rbf_clf_grid = SVC(kernel='rbf', C=params['C'], gamma=params['gamma'], class_weight='balanced')
rbf_clf_grid.fit(grid_Xtrain_small, grid_ytrain_small)
print("Training complete.")
print(f"Training features shape: {grid_Xtrain.shape}")

grid_models = [rbf_clf_grid]
grid_model_names = ['RBF SVM (PCA Grid 10K)']

In [ ]:
# Save the model to a pickle file
if not os.path.exists(path + 'models'):
    os.makedirs(path + 'models')

save_models_to_pickle(grid_models, grid_model_names, 'grid_models/')

In [ ]:
# Load the grid models from the pickle files in the models folder
grid_models = []
grid_model_names = []
for filename in os.listdir(path + 'grid_models/'):
    if filename.endswith('.pkl'):
        model = load_models_from_pickle(path + 'grid_models/' + filename)
        grid_models.append(model)
        grid_model_names.append(filename.replace('.pkl', '').replace('_', ' '))
        print(f"Model loaded from {filename}")
        print(f"Model name: {grid_model_names[-1]}")

NOTE: The ground truth in this section is not based on the actual CSV blobs numbers since we wouldn't be able to make a 1-1 comparison.<br>
Otherwise, the numbers would be even more off than they already are.

In [ ]:
# Run an entire detection on the image using a trained grid model
def do_full_detection_grid(model, grid_pca, image_name):
    
    # Load the original image
    img = cv2.imread(path + 'rgb/' + image_name + '.png', cv2.IMREAD_GRAYSCALE)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    
    # Get the grid features, labels and coords for the image
    features, ground_truth_labels, coords = get_grid_features_and_labels_for_image_parallel(image_name, grid_size=60)
    
    # Standardize the features using the same scalers and PCA as for training
    features = grid_scaler.transform(features)  # Standardize the features using the same scaler as for training
    features = grid_pca.transform(features)  # Apply PCA to the features
    features = grid_pca_scaler.transform(features)  # Standardize the features using the same scaler as for training
    # print(f"Features shape: {features.shape}")
    
    # Predict the class of each grid using the trained model
    predictions = model.predict(features)
    
    ground_truth_image = display_grid_labels_with_pixel_labels(image_name)
    
    display_image = img_rgb.copy()
    
    csv_data = pd.read_csv(path + 'csv/' + image_name + '.csv')
    ground_truth_labels = get_labels_from_csv(csv_data, coords)
    
    ground_truth_tumor_count = np.sum(ground_truth_labels == 1)
    ground_truth_non_tumor_count = np.sum(ground_truth_labels == 2)
    
    predicted_tumor_count = np.sum(predictions == 1)
    predicted_non_tumor_count = np.sum(predictions == 2)
    
    # Draw the grid labels on the image
    for grid_label, coords in zip(predictions, coords):
        xmin, ymin, xmax, ymax = coords
        if grid_label == 1:
            color = (255, 0, 0) # Red for tumor
        elif grid_label == 2:
            color = (0, 0, 255) # Blue for non-tumor
        else:
            color = (0, 255, 0) # Green for unknown
        cv2.rectangle(display_image, (xmin, ymin), (xmax, ymax), color, 1)
    
    return ground_truth_tumor_count, ground_truth_non_tumor_count, predicted_tumor_count, predicted_non_tumor_count, predictions, accuracy_score(ground_truth_labels, predictions), ground_truth_image, display_image

# Test the function with the testing image
testing_image_path = 'TCGA-A2-A0D2-DX1_id-5ea40b0eddda5f8398997d45_left-100214_top-23322_bottom-23602_right-100471' # This image has a mix of tumors and non-tumors

# Run the full detection on the testing image using each model
for model, model_name in zip(grid_models, grid_model_names):
    print("--------------------------------------------")
    print(f"Running full detection with model: {model_name}")
    num_tumors, num_non_tumors, predicted_tumors, predicted_non_tumors, predictions, accuracy, ground_image, predicted_image = do_full_detection_grid(model, grid_pca, testing_image_path)
    print(f"Predictions: {predictions}")
    print(f"Number of tumors (Ground Truth): {num_tumors}")
    print(f"Number of tumors (Predicted): {predicted_tumors}")
    print(f"Number of non-tumors (Ground Truth): {num_non_tumors}")
    print(f"Number of non-tumors (Predicted): {predicted_non_tumors}")
    print(f"Accuracy: {accuracy:.4f}")
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(ground_image)
    plt.title('Ground Truth Pixel Labels')
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.imshow(predicted_image)
    plt.title('Grid Labels\nTumor (Red), Non-tumor (Blue), Unknown (Green)')
    plt.axis('off')
    plt.show()

In [ ]:
# Function to get the classification report and confusion matrix
def report_performance_metrics_grid(model, X_train, y_train, X_val, y_val):

    # Predictions on training data
    y_train_pred = model.predict(X_train)
    
    # Predictions on validation data
    y_val_pred = model.predict(X_val)
    
    # Training metrics
    print("Performance Metrics for Training Data:")
    print("Accuracy:", accuracy_score(y_train, y_train_pred))
    print("Classification Report:")
    print(classification_report(y_train, y_train_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_train, y_train_pred))
    
    # Validation metrics
    print("Performance Metrics for Validation Data:")
    print("Accuracy:", accuracy_score(y_val, y_val_pred))
    print("Classification Report:")
    print(classification_report(y_val, y_val_pred))
    print("Confusion Matrix:")
    confusion_M = confusion_matrix(y_val, y_val_pred)
    print(confusion_M)

    # Display the confusion matrix
    plt.figure(figsize=(5, 5))
    sns.heatmap(confusion_M, annot=True, fmt='d', cmap='Blues', cbar=False, 
                xticklabels=['Unknown', 'Tumor', 'Non-Tumors'], yticklabels=['Unknown', 'Tumor', 'Non-Tumors'])
    plt.xticks()
    plt.title('Linear SVM Confusion Matrix')
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.show()

# Report performance metrics for each model
for i, model in enumerate(grid_models):
    print(f"Performance Metrics for Model {model_names[i]}:")
    report_performance_metrics_grid(model, grid_Xtrain, grid_ytrain, grid_Xval, grid_yval)
    print("\n" + "="*50 + "\n")
    break # Remove this line to run for all models

In [ ]:
# Function to calculate the mean absolute error between ground truth and predicted tumor counts
def calculate_rmse_grid(model, subjects_csv_name):
    predicted_counts = []
    ground_truth_counts = []
    differences = []
    subjects = []

    # Iterate through the validation subject csv
    for i, row in pd.read_csv(subjects_csv_name).iterrows():
        subjects.append(row['subject_id'])
    
    validation_images = []
    for subject in subjects:
        # Get all image names for the subject
        images = get_all_image_names_for_subject(subject)
        validation_images.extend(images)

    for image_name in validation_images:
        
        # Get the ground truth and predicted tumor counts for the image
        ground_truth_tumor_count, _, predicted_tumor_count, _, _, _, _, _ = do_full_detection_grid(model, grid_pca, image_name)
        
        ground_truth_counts.append(ground_truth_tumor_count)
        predicted_counts.append(predicted_tumor_count)
        differences.append(predicted_tumor_count - ground_truth_tumor_count)
    
    # Calculate RMSE between ground truth and predicted counts
    rmse = np.sqrt(mean_squared_error(ground_truth_counts, predicted_counts))
    
    # Print the final RMSE culmination
    print("Final RMSE culmination:", rmse)
    
    return rmse, differences, predicted_counts, ground_truth_counts

# Display the RMSE for each model
for i, model in enumerate(grid_models):
    print(f"RMSE for Model {grid_model_names[i]}:")
    rmse, difference, _, _ = calculate_rmse_grid(model, path+'splits/validate_subjects.csv')
    print(f"RMSE: {rmse:.4f}")
    print("\n" + "="*50 + "\n")
    
    # Plot the differences as a bar chart x axis should be the difference and y axis should be the number of instances
    plt.figure(figsize=(10, 5))
    plt.hist(difference, bins=20, color='blue', alpha=0.7)
    plt.title(f'Differences in Tumor Counts for Model {model_names[i]}')
    plt.xlabel('Difference in Tumor Counts')
    plt.ylabel('Number of Instances')
    plt.grid(axis='y', alpha=0.75)
    plt.axvline(x=0, color='red', linestyle='--', label='Ground Truth')
    plt.legend()
    plt.tight_layout()
    plt.show()

#### 3.2.4. Based on the features you used in the detection model, what pathological structures are easier to segregate from tumors? Justify your answer using appropriate metrics.


In [ ]:
#Extract non-tumor subclasses labels
def extract_subclass_labels(csv_data, subclass_map=None):
    """
    Map raw_classification labels to numeric subclass IDs.
    """
    # Create a copy to not modify the original DataFrame
    csv_data_copy = csv_data.copy()

    # Initialize subclass_map if not provided
    if subclass_map is None:
        subclass_map = {}

    # Get current max subclass ID
    current_max_id = max(subclass_map.values(), default=-1)

    # Map raw_classification to numeric subclass IDs
    #subclass_map = {label: idx for idx, 
    #                label in enumerate(sorted(csv_data_copy['raw_classification'].unique()))}

    # Find new labels
    new_labels = [label for label in csv_data_copy['raw_classification'].unique() 
                  if label not in subclass_map]
    
    # Add new labels to the subclass_map
    for label in new_labels:
        current_max_id += 1
        subclass_map[label] = current_max_id
    
    csv_data_copy['subclass_id'] = csv_data_copy['raw_classification'].map(subclass_map)
    return subclass_map, csv_data_copy

In [ ]:
# Assign subclass labels to blobs based on the bounding box coordinates
def assign_subclass_labels_to_blobs(csv_data, blob_coords):
    subclass_labels = []
    match_count = 0

    for (xmin, ymin, xmax, ymax, coords_x, coords_y) in blob_coords:
        # blob matches the bounding box in the CSV
        #for (xmin, ymin, xmax, ymax) in blob_coords:
        for _, row in csv_data.iterrows():
            
            match = csv_data[
                (csv_data['xmin'] == xmin) & 
                (csv_data['ymin'] == ymin) & 
                (csv_data['xmax'] == (xmax - 1)) &
                (csv_data['ymax'] == (ymax - 1))
            ]
            #print(match)

            if not match.empty:
                #subclass_labels.append(row['subclass_id'])
                match_count += 1
                subclass_labels.append(match.iloc[0]['subclass_id'])
                break
            else:
                subclass_labels.append(-1) # No match found
    
    #print(f"Number of matches found: {match_count}")
    
    #return subclass_labels
    return np.array(subclass_labels) # Convert to numpy array for consistency

In [ ]:
# Calculate precision, recall, and F1 score for each subclass
def calculate_subclass_metrics_aggregated(global_metrics):
    """Calculate precision, recall, and F1 score for each subclass."""
    aggregated_metrics = {}
    for subclass, metrics in global_metrics.items():
        tp = metrics['tp']
        fp = metrics['fp']
        fn = metrics['fn']

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        aggregated_metrics[subclass] = {
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        }
    return aggregated_metrics


def analyze_subclass_performance_for_chunk(model, pca, image_names_chunk, global_subclass_map):
    """Analyze subclass performance for a chunk of images."""
    local_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    for image_name in image_names_chunk:
        # Load the CSV data for the image
        csv_data = pd.read_csv(path + 'csv/' + image_name + '.csv')

        # Extract subclass labels
        subclass_map, csv_data_with_subclass = extract_subclass_labels(csv_data, global_subclass_map)

        # Perform full detection to get predictions
        _, _, predictions, _, blob_coords, _, _, _ = do_full_detection(model, pca, image_name)

        # Assign subclass labels to blobs
        y_true = assign_subclass_labels_to_blobs(csv_data_with_subclass, blob_coords)

        # Filter out unmatched blobs
        valid_indices = y_true != -1
        y_true = y_true[valid_indices]
        predictions = predictions[valid_indices]

        # Update local metrics for each subclass
        for subclass, subclass_id in subclass_map.items():
            y_true_binary = (y_true == subclass_id).astype(int)
            y_pred_binary = (predictions == subclass_id).astype(int)

            tp = (y_true_binary & y_pred_binary).sum()
            fp = (y_pred_binary & ~y_true_binary).sum()
            fn = (y_true_binary & ~y_pred_binary).sum()

            local_metrics[subclass]['tp'] += tp
            local_metrics[subclass]['fp'] += fp
            local_metrics[subclass]['fn'] += fn

    return local_metrics


def merge_metrics(global_metrics, local_metrics):
    """Merge local metrics into global metrics."""
    for subclass, metrics in local_metrics.items():
        global_metrics[subclass]['tp'] += metrics['tp']
        global_metrics[subclass]['fp'] += metrics['fp']
        global_metrics[subclass]['fn'] += metrics['fn']


def analyze_subclass_performance_aggregated_parallel(model, pca, image_names, global_subclass_map, n_jobs=-1):
    """Analyze subclass performance across all images in parallel."""
    print("Analyzing subclass performance across all images in parallel...")

    # Split the image names into chunks for parallel processing
    num_chunks = max(1, len(image_names) // os.cpu_count())
    image_chunks = [image_names[i:i + num_chunks] for i in range(0, len(image_names), num_chunks)]

    # Process each chunk in parallel
    results = Parallel(n_jobs=n_jobs)(
        delayed(analyze_subclass_performance_for_chunk)(model, pca, chunk, global_subclass_map)
        for chunk in image_chunks
    )

    # Combine the results from all chunks
    global_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    for local_metrics in results:
        merge_metrics(global_metrics, local_metrics)

    # Calculate aggregated metrics
    aggregated_metrics = calculate_subclass_metrics_aggregated(global_metrics)

    return aggregated_metrics, global_subclass_map


# Step 5: Run analysis on all images
image_names = [os.path.splitext(img_name)[0] for img_name in os.listdir(path + 'rgb')]

# Run the analysis for each model
for model, model_name in zip(models, model_names):
    global_subclass_map = {}

    print(f"Analyzing subclass performance for model: {model_name}")
    aggregated_metrics, global_subclass_map = analyze_subclass_performance_aggregated_parallel(
        model, pca, image_names, global_subclass_map, n_jobs=-1
    )

    for subclass, metrics in aggregated_metrics.items():
        print(f"Subclass: {subclass}")
        print(f"  Precision: {metrics['precision']:.4f}")
        print(f"  Recall: {metrics['recall']:.4f}")
        print(f"  F1 Score: {metrics['f1_score']:.4f}")
        print("-" * 30)

    print("=" * 50)

## 4. Bonus Question

### 4.1. Supervised Learning: Using any pre-trained deep learning model, fine-tune it to segment any random 50 samples. Report the performance on training data. Include key details of the implementation.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg # Not super necessary
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer

import cv2
import numpy as np, sys
import math as m
import os
import pandas as pd
import ast 
from glob import glob
import tensorflow as tf
import matplotlib.pyplot as plt
from keras import layers, models
from sklearn.model_selection import train_test_split

path = './QC/'

In [ ]:


def get_class_mapping(csv_dir):
    class_map = {}
    class_id=1
    for csv_file in os.listdir(csv_dir):
        if csv_file.endswith('.csv'):
            df = pd.read_csv(os.path.join(csv_dir, csv_file))
            for cls in df["raw_classification"].unique():
                if cls not in class_map:
                    class_map[cls] = class_id
                    class_id += 1
    class_map["background"] = 0
    return class_map

def get_class_labels(class_map):
    return {v: k for k, v in class_map.items()}

csv_path= os.path.join(path, "csv")
CLASS_MAP = get_class_mapping(csv_path)
nb_classes = 10
class_labels = get_class_labels(CLASS_MAP)

print("Class mapping: ", CLASS_MAP)
print("Class labels: ", class_labels)
print("Number of classes: ", nb_classes)

rgb_dir = os.path.join(path, "rgb")
csv_dir = os.path.join(path, "csv")

def draw_shapes_from_csv(input_csv, target_height, target_width, original_height, original_width, class_map):
    
    df = pd.read_csv(input_csv)
    mask = np.zeros((target_height, target_width), dtype=np.uint8)

    scale_x = target_width / original_width
    scale_y = target_height / original_height

    for _, row in df.iterrows():  
        
        class_id = class_map.get(row["raw_classification"], 0)
        
        print(f"Class: {row['raw_classification']}, Assigned Class ID: {class_id}")
        
        coords_x = np.array(ast.literal_eval(row['coords_x']), dtype=np.float32) * scale_x
        coords_y = np.array(ast.literal_eval(row['coords_y']), dtype=np.float32) * scale_y
        points = np.column_stack((coords_x, coords_y)).astype(np.int32)

        if row["type"] == "polyline":
            cv2.fillPoly(mask, [points], color=class_id)
        elif row["type"] == "rectangle":
            x0, y0 = int(coords_x[0]), int(coords_y[0])
            x2, y2 = int(coords_x[2]), int(coords_y[2])
            cv2.rectangle(mask, (x0, y0), (x2, y2), color=class_id, thickness=-1)
    return mask


def load_data_from_csv(rgb_dir, csv_dir, count=50, target_size=(128, 128)):
    images, masks = [], []
    rgb_files = sorted(os.listdir(rgb_dir))[:count]

    for fname in rgb_files:
        print (f"Processing file: {fname}")
        img_path = os.path.join(rgb_dir, fname)
        csv_path = os.path.join(csv_dir, fname.replace('.png', '.csv'))

        img = cv2.imread(img_path)


        h_orig, w_orig = img.shape[:2]  # Original size
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, target_size)
        h_resized, w_resized = target_size

        mask = draw_shapes_from_csv(csv_path, h_resized, w_resized, h_orig, w_orig, CLASS_MAP)
        
        mask_one_hot = tf.keras.utils.to_categorical(mask, num_classes=nb_classes)
        
        print(f"Unique values in mask: {np.unique(masks)}")
        print(f"Unique values in mask_one_hot: {np.unique(mask_one_hot)}")
        
        masks.append(mask_one_hot.astype(np.uint8)) 

        images.append(img / 255.0)

    return np.array(images, dtype=np.float32),np.array(masks, dtype=np.uint8)

images, masks = load_data_from_csv(rgb_dir, csv_dir, count=50)
print(f"Loaded {len(images)} images and masks from CSV")

BATCH_SIZE = 8
BUFFER_SIZE = 100

X_train, X_test, y_train, y_test = train_test_split(images, masks, test_size=8, random_state=42)

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(1)

def build_unet_model(nb_classes):
    def conv_block(x, filters):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        return x

    def down_block(x, filters):
        f = conv_block(x, filters)
        p = layers.MaxPool2D(2)(f)
        return f, p

    def up_block(x, skip, filters):
        x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same')(x)
        x = layers.concatenate([x, skip])
        return conv_block(x, filters)

    inputs = layers.Input((128, 128, 3))
    f1, p1 = down_block(inputs, 64)
    f2, p2 = down_block(p1, 128)
    f3, p3 = down_block(p2, 256)
    f4, p4 = down_block(p3, 512)

    bottleneck = conv_block(p4, 1024)

    u1 = up_block(bottleneck, f4, 512)
    u2 = up_block(u1, f3, 256)
    u3 = up_block(u2, f2, 128)
    u4 = up_block(u3, f1, 64)

    outputs = layers.Conv2D(nb_classes, 1, activation='softmax')(u4)
    return models.Model(inputs, outputs)

model = build_unet_model(nb_classes)

unique, counts = np.unique(y_train, return_counts=True)
print("Class Distribution in Masks:", dict(zip(unique, counts)))

flat_labels = np.argmax(y_train, axis=-1).flatten()
unique, counts = np.unique(flat_labels, return_counts=True)
class_weights = {i: max(counts) / counts[i] for i in unique}
print("Class Weights:", class_weights)

loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False)

def weighted_categorical_crossentropy(y_true, y_pred):
    class_weights_tensor = tf.constant(list(class_weights.values()), dtype=tf.float32)
    weights = tf.reduce_sum(class_weights_tensor * y_true, axis=-1)
    return tf.keras.losses.categorical_crossentropy(y_true, y_pred) * weights

model.compile(optimizer='adam', loss=weighted_categorical_crossentropy, metrics=['categorical_accuracy'])



EPOCHS = 15
steps_per_epoch = len(images) // BATCH_SIZE

history =model.fit(train_ds, epochs=EPOCHS, validation_data=test_ds)


In [ ]:


def create_mask(pred):
    return np.argmax(pred[0], axis=-1).astype(np.uint8)

def show_prediction(image, true_mask, pred_mask, class_labels):
    print(f"pred 2 shape: {pred_mask.shape}")
    
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(image)
    ax[0].set_title("Input Image")
    
    # Convert the true mask from one-hot encoding to a single-channel mask
    true_mask = np.squeeze(np.argmax(true_mask, axis=-1))  # Convert to (128, 128)
    ax[1].imshow(true_mask, cmap=plt.cm.get_cmap('jet', nb_classes), vmin=0, vmax=nb_classes-1)
    ax[1].set_title("Ground Truth Mask")
    
    print(f"Predicted mask shape 3: {pred_mask.shape}")
    pred_mask_single_channel = np.squeeze(pred_mask)  # Convert to (128, 128)
    print(f"Predicted mask shape 4: {pred_mask_single_channel.shape}")
    ax[2].imshow(pred_mask_single_channel, cmap=plt.cm.get_cmap('jet', nb_classes), vmin=0, vmax=nb_classes-1)
    ax[2].set_title("Predicted Mask")
    
    mask_uint8 = pred_mask_single_channel.astype(np.uint8)

    for class_id in np.unique(mask_uint8):
        if class_id == 0:
            continue

        mask_class = (mask_uint8 == class_id).astype(np.uint8)
        contours, _ = cv2.findContours(mask_class, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for contour in contours:
            ax[2].plot(contour[:, 0, 0], contour[:, 0, 1], color='white', linewidth=1.5)

        y, x = np.where(mask_class)
        if len(x) > 0 and len(y) > 0:
            centroid_x = int(np.mean(x))
            centroid_y = int(np.mean(y))
            ax[2].text(centroid_x, centroid_y, class_labels.get(class_id, 'Unknown'), color='white', fontsize=10, ha='center', va='center', bbox=dict(facecolor='black', alpha=0.5))
            
    for a in ax:
        a.axis('off')
    plt.tight_layout()
    plt.show()


for i in range(3):
    sample_img = np.expand_dims(X_test[i], axis=0)
    prediction = model.predict(sample_img)
    print(f"Prediction shape 1: {prediction.shape}")
    print(f"Prediction range: {prediction.min():.4f} - {prediction.max():.4f}")
    print(f"Unique values in argmax(prediction): {np.unique(np.argmax(prediction, axis=-1))}")
    print("Unique values in masks:", np.unique(masks))
    show_prediction(X_test[i], y_test[i], create_mask(prediction), class_labels)


### 4.2. 2. Inference: Report the performance of this segmentor on the remaining 8 samples.

In [ ]:
from sklearn.model_selection import train_test_split

rgb_dir = os.path.join(path, "rgb")
csv_dir = os.path.join(path, "csv")
TARGET_SIZE = (128, 128)

images, masks = load_data_from_csv(rgb_dir, csv_dir, count=50, target_size=TARGET_SIZE)
print(f"Loaded {len(images)} samples.")

X_train, X_test, y_train, y_test = train_test_split(images, masks, test_size=8, random_state=42)

BATCH_SIZE = 8
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(100).batch(BATCH_SIZE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(1)

def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)


model = build_unet_model(nb_classes)
model.compile(optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', dice_coef])

model.fit(train_ds, epochs=15)

loss, acc, dice = model.evaluate(test_ds)
print("\n Evaluation on 8 held-out samples:")
print(f" - Loss: {loss:.4f}")
print(f" - Accuracy: {acc:.4f}")
print(f" - DICE Coefficient: {dice:.4f}")


In [ ]:


def show_prediction(image, true_mask, pred_mask, class_labels):
    print(f"pred 2 shape: {pred_mask.shape}")
    
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(image)
    ax[0].set_title("Input Image")
    
    # Convert the true mask from one-hot encoding to a single-channel mask
    true_mask = np.squeeze(np.argmax(true_mask, axis=-1))  # Convert to (128, 128)
    ax[1].imshow(true_mask, cmap=plt.cm.get_cmap('jet', nb_classes), vmin=0, vmax=nb_classes-1)
    ax[1].set_title("Ground Truth Mask")
    
    print(f"Predicted mask shape 3: {pred_mask.shape}")
    pred_mask_single_channel = np.squeeze(pred_mask)  # Convert to (128, 128)
    print(f"Predicted mask shape 4: {pred_mask_single_channel.shape}")
    ax[2].imshow(pred_mask_single_channel, cmap=plt.cm.get_cmap('jet', nb_classes), vmin=0, vmax=nb_classes-1)
    ax[2].set_title("Predicted Mask")
    
    mask_uint8 = pred_mask_single_channel.astype(np.uint8)

    for class_id in np.unique(mask_uint8):
        if class_id == 0:
            continue

        mask_class = (mask_uint8 == class_id).astype(np.uint8)
        contours, _ = cv2.findContours(mask_class, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for contour in contours:
            ax[2].plot(contour[:, 0, 0], contour[:, 0, 1], color='white', linewidth=1.5)

        y, x = np.where(mask_class)
        if len(x) > 0 and len(y) > 0:
            centroid_x = int(np.mean(x))
            centroid_y = int(np.mean(y))
            ax[2].text(centroid_x, centroid_y, class_labels.get(class_id, 'Unknown'), color='white', fontsize=10, ha='center', va='center', bbox=dict(facecolor='black', alpha=0.5))
            
    for a in ax:
        a.axis('off')
    plt.tight_layout()
    plt.show()


for i in range(3):
    sample_img = np.expand_dims(X_test[i], axis=0)
    prediction = model.predict(sample_img)
    print(f"Prediction shape 1: {prediction.shape}")
    print(f"Prediction range: {prediction.min():.4f} - {prediction.max():.4f}")
    print(f"Unique values in argmax(prediction): {np.unique(np.argmax(prediction, axis=-1))}")
    print("Unique values in masks:", np.unique(masks))
    show_prediction(X_test[i], y_test[i], create_mask(prediction), class_labels)
